In [2]:
# 1. Install SAHI and Ultralytics
!pip install -U sahi ultralytics



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 11.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python
    Found existing installation: opencv-python 4.13.0.92
    Uninstalling opencv-python-4.13.0.92:
      Successfully uninstalled opencv-python-4.13.0.92


In [ ]:
!uv pip install ultralytics sahi
import ultralytics
from ultralytics.utils.downloads import safe_download
ultralytics.checks()

Ultralytics 8.4.36 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.3/235.7 GB disk)


In [ ]:
# Clone ultralytics repo
!git clone https://github.com/ultralytics/ultralytics

Cloning into 'ultralytics'...
remote: Enumerating objects: 86157, done.
remote: Counting objects: 100% (276/276), done.
remote: Compressing objects: 100% (190/190), done.
remote: Total 86157 (delta 183), reused 95 (delta 86), pack-reused 85881 (from 3)
Receiving objects: 100% (86157/86157), 46.94 MiB | 13.71 MiB/s, done.
Resolving deltas: 100% (64943/64943), done.


# YOLO V8 + SAHI

##Trainings

###Subset: Full Dataset

In [2]:
import os
import numpy as np
import pandas as pd
import yaml
import time
import cv2
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import visualize_object_predictions, read_image

# --- CONFIGURATION ---
BASE_PATH = '/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data'
OUTPUT_VIS_DIR = '/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV8/full_dataset/visuals'
FOLDERS = ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']
SMALL_OBJ_THRESHOLD = 32 * 32
os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)

MODEL_MAP = {
    'fold_0': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/full_dataset/fold_0/weights/best.pt',
    'fold_1': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/full_dataset/fold_1/weights/best.pt',
    'fold_2': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/full_dataset/fold_2/weights/best.pt',
    'fold_3': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/full_dataset/fold_3/weights/best.pt',
    'fold_4': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/full_dataset/fold_4/weights/best.pt',
}

# --- HELPER FUNCTIONS ---
def get_image_dimensions(image_path):
    with Image.open(image_path) as img: return img.size

def load_yaml(yaml_path):
    with open(yaml_path, 'r') as f: return yaml.safe_load(f)

def bbox_iou(box1, box2):
    x1, y1, x2, y2 = max(box1[0], box2[0]), max(box1[1], box2[1]), min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def load_yolo_annotations(label_dir, image_filename, image_dims):
    annotations = []
    label_path = os.path.join(label_dir, os.path.splitext(image_filename)[0] + '.txt')
    if not os.path.exists(label_path): return annotations
    img_w, img_h = image_dims
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                c_id, cx, cy, w, h = map(float, parts)
                xmin, ymin = int((cx - w/2) * img_w), int((cy - h/2) * img_h)
                xmax, ymax = int((cx + w/2) * img_w), int((cy + h/2) * img_h)
                annotations.append({'bbox': [xmin, ymin, xmax, ymax], 'category_id': int(c_id), 'area': (xmax-xmin)*(ymax-ymin)})
    return annotations

def calculate_ap(preds, gts, iou_threshold=0.5):
    if not gts: return 0.0
    if not preds: return 0.0
    preds = sorted(preds, key=lambda x: x['score'], reverse=True)
    tp, fp = np.zeros(len(preds)), np.zeros(len(preds))
    matched_gt = [False] * len(gts)
    for i, p in enumerate(preds):
        best_iou, best_idx = -1, -1
        for j, g in enumerate(gts):
            if not matched_gt[j] and p['category_id'] == g['category_id']:
                iou = bbox_iou(p['bbox'], g['bbox'])
                if iou > best_iou: best_iou, best_idx = iou, j
        if best_iou >= iou_threshold:
            tp[i] = 1
            matched_gt[best_idx] = True
        else: fp[i] = 1
    tp_cumsum, fp_cumsum = np.cumsum(tp), np.cumsum(fp)
    recalls = tp_cumsum / len(gts)
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)
    # 11-point interpolation
    ap = 0.0
    for t in np.arange(0, 1.1, 0.1):
        p = np.max(precisions[recalls >= t]) if any(recalls >= t) else 0.0
        ap += p / 11.0
    return ap

# --- MAIN EVALUATION LOOP ---
data_yaml = load_yaml(f"{BASE_PATH}/fold_0/data.yaml")
num_classes = len(data_yaml.get('names', []))
results_log = []

for folder in FOLDERS:
    print(f"\n--- 🚀 Starting Evaluation with mAP: {folder} ---")
    img_dir, label_dir = f"{BASE_PATH}/{folder}/valid/images", f"{BASE_PATH}/{folder}/valid/labels"
    image_filenames = [f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    detection_model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics', model_path=MODEL_MAP[folder],
        confidence_threshold=0.30, device='cuda:0'
    )

    f_tp, f_fp, f_fn = 0, 0, 0
    f_tp_s, f_fn_s = 0, 0
    all_preds_cls = {i: [] for i in range(num_classes)}
    all_gts_cls = {i: [] for i in range(num_classes)}

    for idx, img_name in enumerate(image_filenames):
        image_path = os.path.join(img_dir, img_name)
        dims = get_image_dimensions(image_path)
        gts = load_yolo_annotations(label_dir, img_name, dims)

        result = get_sliced_prediction(
            image_path, detection_model,
            slice_height=1024, slice_width=1024,
            overlap_height_ratio=0.2, overlap_width_ratio=0.2,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.3
        )

        preds = [{'bbox': obj.bbox.to_xyxy(), 'category_id': obj.category.id, 'score': obj.score.value}
                 for obj in result.object_prediction_list]

        # Simple counts for Precision/Recall
        matched_gt = [False] * len(gts)
        sorted_p = sorted(preds, key=lambda x: x['score'], reverse=True)
        for p in sorted_p:
            best_iou, best_idx = -1, -1
            for i, g in enumerate(gts):
                if not matched_gt[i] and p['category_id'] == g['category_id']:
                    iou = bbox_iou(p['bbox'], g['bbox'])
                    if iou > best_iou: best_iou, best_idx = iou, i
            if best_iou >= 0.5:
                f_tp += 1
                matched_gt[best_idx] = True
            else: f_fp += 1
        f_fn += matched_gt.count(False)

        # Store for mAP and Small Object metrics
        for p in preds: all_preds_cls[p['category_id']].append(p)
        for g in gts:
            all_gts_cls[g['category_id']].append(g)
            if g['area'] < SMALL_OBJ_THRESHOLD:
                # Local check for small recall
                best_iou = max([bbox_iou(p['bbox'], g['bbox']) for p in preds if p['category_id'] == g['category_id']] + [0])
                if best_iou >= 0.5: f_tp_s += 1
                else: f_fn_s += 1

        if idx < 3: # Save samples
            visual_result = visualize_object_predictions(np.array(read_image(image_path)), result.object_prediction_list)
            cv2.imwrite(os.path.join(OUTPUT_VIS_DIR, f"{folder}_{img_name}"), cv2.cvtColor(visual_result['image'], cv2.COLOR_RGB2BGR))

    # Fold Metrics
    p_f = f_tp / (f_tp + f_fp) if (f_tp + f_fp) > 0 else 0
    r_f = f_tp / (f_tp + f_fn) if (f_tp + f_fn) > 0 else 0
    f1_f = 2*(p_f*r_f)/(p_f+r_f) if (p_f+r_f)>0 else 0
    map50 = np.mean([calculate_ap(all_preds_cls[i], all_gts_cls[i]) for i in range(num_classes) if all_gts_cls[i]])
    s_rec = f_tp_s / (f_tp_s + f_fn_s) if (f_tp_s + f_fn_s) > 0 else 0

    results_log.append({
        'Folder': folder, 'mAP@0.5': round(map50, 4), 'Precision': round(p_f, 4),
        'Recall': round(r_f, 4), 'F1': round(f1_f, 4), 'Small_Recall': round(s_rec, 4)
    })

df = pd.DataFrame(results_log)
avg_row = df.mean(numeric_only=True).to_dict(); avg_row['Folder'] = 'AVERAGE'
df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
print("\n--- Final Results (SAHI + mAP) ---")
print(df.to_string(index=False))


--- 🚀 Starting Evaluation with mAP: fold_0 ---
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Performing prediction on 9 slices.
Performing prediction on 40 slices.
Performing prediction on 6 slices.
Performing prediction on 3 slices.
Performing prediction on 3 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 9 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 20 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performin

###Subset: Components Only

In [3]:
import os
import numpy as np
import pandas as pd
import yaml
import time
import cv2
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import visualize_object_predictions, read_image

# --- CONFIGURATION ---
BASE_PATH = '/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data'
OUTPUT_VIS_DIR = '/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV8/components_only/visuals'
FOLDERS = ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']
SMALL_OBJ_THRESHOLD = 32 * 32
os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)

MODEL_MAP = {
    'fold_0': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/components_only/fold_0/weights/best.pt',
    'fold_1': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/components_only/fold_1/weights/best.pt',
    'fold_2': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/components_only/fold_2/weights/best.pt',
    'fold_3': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/components_only/fold_3/weights/best.pt',
    'fold_4': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/components_only/fold_4/weights/best.pt',
}

# --- HELPER FUNCTIONS ---
def get_image_dimensions(image_path):
    with Image.open(image_path) as img: return img.size

def load_yaml(yaml_path):
    with open(yaml_path, 'r') as f: return yaml.safe_load(f)

def bbox_iou(box1, box2):
    x1, y1, x2, y2 = max(box1[0], box2[0]), max(box1[1], box2[1]), min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def load_yolo_annotations(label_dir, image_filename, image_dims):
    annotations = []
    label_path = os.path.join(label_dir, os.path.splitext(image_filename)[0] + '.txt')
    if not os.path.exists(label_path): return annotations
    img_w, img_h = image_dims
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                c_id, cx, cy, w, h = map(float, parts)
                xmin, ymin = int((cx - w/2) * img_w), int((cy - h/2) * img_h)
                xmax, ymax = int((cx + w/2) * img_w), int((cy + h/2) * img_h)
                annotations.append({'bbox': [xmin, ymin, xmax, ymax], 'category_id': int(c_id), 'area': (xmax-xmin)*(ymax-ymin)})
    return annotations

def calculate_ap(preds, gts, iou_threshold=0.5):
    if not gts: return 0.0
    if not preds: return 0.0
    preds = sorted(preds, key=lambda x: x['score'], reverse=True)
    tp, fp = np.zeros(len(preds)), np.zeros(len(preds))
    matched_gt = [False] * len(gts)
    for i, p in enumerate(preds):
        best_iou, best_idx = -1, -1
        for j, g in enumerate(gts):
            if not matched_gt[j] and p['category_id'] == g['category_id']:
                iou = bbox_iou(p['bbox'], g['bbox'])
                if iou > best_iou: best_iou, best_idx = iou, j
        if best_iou >= iou_threshold:
            tp[i] = 1
            matched_gt[best_idx] = True
        else: fp[i] = 1
    tp_cumsum, fp_cumsum = np.cumsum(tp), np.cumsum(fp)
    recalls = tp_cumsum / len(gts)
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)
    # 11-point interpolation
    ap = 0.0
    for t in np.arange(0, 1.1, 0.1):
        p = np.max(precisions[recalls >= t]) if any(recalls >= t) else 0.0
        ap += p / 11.0
    return ap

# --- MAIN EVALUATION LOOP ---
data_yaml = load_yaml(f"{BASE_PATH}/fold_0/data.yaml")
num_classes = len(data_yaml.get('names', []))
results_log = []

for folder in FOLDERS:
    print(f"\n--- 🚀 Starting Evaluation with mAP: {folder} ---")
    img_dir, label_dir = f"{BASE_PATH}/{folder}/valid/images", f"{BASE_PATH}/{folder}/valid/labels"
    image_filenames = [f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    detection_model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics', model_path=MODEL_MAP[folder],
        confidence_threshold=0.30, device='cuda:0'
    )

    f_tp, f_fp, f_fn = 0, 0, 0
    f_tp_s, f_fn_s = 0, 0
    all_preds_cls = {i: [] for i in range(num_classes)}
    all_gts_cls = {i: [] for i in range(num_classes)}

    for idx, img_name in enumerate(image_filenames):
        image_path = os.path.join(img_dir, img_name)
        dims = get_image_dimensions(image_path)
        gts = load_yolo_annotations(label_dir, img_name, dims)

        result = get_sliced_prediction(
            image_path, detection_model,
            slice_height=1024, slice_width=1024,
            overlap_height_ratio=0.2, overlap_width_ratio=0.2,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.3
        )

        preds = [{'bbox': obj.bbox.to_xyxy(), 'category_id': obj.category.id, 'score': obj.score.value}
                 for obj in result.object_prediction_list]

        # Simple counts for Precision/Recall
        matched_gt = [False] * len(gts)
        sorted_p = sorted(preds, key=lambda x: x['score'], reverse=True)
        for p in sorted_p:
            best_iou, best_idx = -1, -1
            for i, g in enumerate(gts):
                if not matched_gt[i] and p['category_id'] == g['category_id']:
                    iou = bbox_iou(p['bbox'], g['bbox'])
                    if iou > best_iou: best_iou, best_idx = iou, i
            if best_iou >= 0.5:
                f_tp += 1
                matched_gt[best_idx] = True
            else: f_fp += 1
        f_fn += matched_gt.count(False)

        # Store for mAP and Small Object metrics
        for p in preds: all_preds_cls[p['category_id']].append(p)
        for g in gts:
            all_gts_cls[g['category_id']].append(g)
            if g['area'] < SMALL_OBJ_THRESHOLD:
                # Local check for small recall
                best_iou = max([bbox_iou(p['bbox'], g['bbox']) for p in preds if p['category_id'] == g['category_id']] + [0])
                if best_iou >= 0.5: f_tp_s += 1
                else: f_fn_s += 1

        if idx < 3: # Save samples
            visual_result = visualize_object_predictions(np.array(read_image(image_path)), result.object_prediction_list)
            cv2.imwrite(os.path.join(OUTPUT_VIS_DIR, f"{folder}_{img_name}"), cv2.cvtColor(visual_result['image'], cv2.COLOR_RGB2BGR))

    # Fold Metrics
    p_f = f_tp / (f_tp + f_fp) if (f_tp + f_fp) > 0 else 0
    r_f = f_tp / (f_tp + f_fn) if (f_tp + f_fn) > 0 else 0
    f1_f = 2*(p_f*r_f)/(p_f+r_f) if (p_f+r_f)>0 else 0
    map50 = np.mean([calculate_ap(all_preds_cls[i], all_gts_cls[i]) for i in range(num_classes) if all_gts_cls[i]])
    s_rec = f_tp_s / (f_tp_s + f_fn_s) if (f_tp_s + f_fn_s) > 0 else 0

    results_log.append({
        'Folder': folder, 'mAP@0.5': round(map50, 4), 'Precision': round(p_f, 4),
        'Recall': round(r_f, 4), 'F1': round(f1_f, 4), 'Small_Recall': round(s_rec, 4)
    })

df = pd.DataFrame(results_log)
avg_row = df.mean(numeric_only=True).to_dict(); avg_row['Folder'] = 'AVERAGE'
df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
print("\n--- Final Results (SAHI + mAP) ---")
print(df.to_string(index=False))


--- 🚀 Starting Evaluation with mAP: fold_0 ---
Performing prediction on 6 slices.
Performing prediction on 9 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 9 slices.
Performing prediction on 40 slices.
Performing prediction on 6 slices.
Performing prediction on 3 slices.
Performing prediction on 6 slices.
Performing prediction on 3 slices.
Performing prediction on 1 slices.
Performing prediction on 4 slices.
Performing prediction on 20 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 20 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 20 slices.
Performing prediction on 6 slices.
Performing prediction on 35 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Pe

###Subset: Missing Only

In [ ]:
import os
import numpy as np
import pandas as pd
import yaml
import time
import cv2
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import visualize_object_predictions, read_image

# --- CONFIGURATION ---
BASE_PATH = '/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data'
OUTPUT_VIS_DIR = '/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV8/missing_only/visuals'
FOLDERS = ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']
SMALL_OBJ_THRESHOLD = 32 * 32
os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)

MODEL_MAP = {
    'fold_0': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/missing_only/fold_0/weights/best.pt',
    'fold_1': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/missing_only/fold_1/weights/best.pt',
    'fold_2': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/missing_only/fold_2/weights/best.pt',
    'fold_3': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/missing_only/fold_3/weights/best.pt',
    'fold_4': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/missing_only/fold_4/weights/best.pt',
}

# --- HELPER FUNCTIONS ---
def get_image_dimensions(image_path):
    with Image.open(image_path) as img: return img.size

def load_yaml(yaml_path):
    with open(yaml_path, 'r') as f: return yaml.safe_load(f)

def bbox_iou(box1, box2):
    x1, y1, x2, y2 = max(box1[0], box2[0]), max(box1[1], box2[1]), min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def load_yolo_annotations(label_dir, image_filename, image_dims):
    annotations = []
    label_path = os.path.join(label_dir, os.path.splitext(image_filename)[0] + '.txt')
    if not os.path.exists(label_path): return annotations
    img_w, img_h = image_dims
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                c_id, cx, cy, w, h = map(float, parts)
                xmin, ymin = int((cx - w/2) * img_w), int((cy - h/2) * img_h)
                xmax, ymax = int((cx + w/2) * img_w), int((cy + h/2) * img_h)
                annotations.append({'bbox': [xmin, ymin, xmax, ymax], 'category_id': int(c_id), 'area': (xmax-xmin)*(ymax-ymin)})
    return annotations

def calculate_ap(preds, gts, iou_threshold=0.5):
    if not gts: return 0.0
    if not preds: return 0.0
    preds = sorted(preds, key=lambda x: x['score'], reverse=True)
    tp, fp = np.zeros(len(preds)), np.zeros(len(preds))
    matched_gt = [False] * len(gts)
    for i, p in enumerate(preds):
        best_iou, best_idx = -1, -1
        for j, g in enumerate(gts):
            if not matched_gt[j] and p['category_id'] == g['category_id']:
                iou = bbox_iou(p['bbox'], g['bbox'])
                if iou > best_iou: best_iou, best_idx = iou, j
        if best_iou >= iou_threshold:
            tp[i] = 1
            matched_gt[best_idx] = True
        else: fp[i] = 1
    tp_cumsum, fp_cumsum = np.cumsum(tp), np.cumsum(fp)
    recalls = tp_cumsum / len(gts)
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)
    # 11-point interpolation
    ap = 0.0
    for t in np.arange(0, 1.1, 0.1):
        p = np.max(precisions[recalls >= t]) if any(recalls >= t) else 0.0
        ap += p / 11.0
    return ap

# --- MAIN EVALUATION LOOP ---
data_yaml = load_yaml(f"{BASE_PATH}/fold_0/data.yaml")
num_classes = len(data_yaml.get('names', []))
results_log = []

for folder in FOLDERS:
    print(f"\n--- 🚀 Starting Evaluation with mAP: {folder} ---")
    img_dir, label_dir = f"{BASE_PATH}/{folder}/valid/images", f"{BASE_PATH}/{folder}/valid/labels"
    image_filenames = [f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    detection_model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics', model_path=MODEL_MAP[folder],
        confidence_threshold=0.30, device='cuda:0'
    )

    f_tp, f_fp, f_fn = 0, 0, 0
    f_tp_s, f_fn_s = 0, 0
    all_preds_cls = {i: [] for i in range(num_classes)}
    all_gts_cls = {i: [] for i in range(num_classes)}

    for idx, img_name in enumerate(image_filenames):
        image_path = os.path.join(img_dir, img_name)
        dims = get_image_dimensions(image_path)
        gts = load_yolo_annotations(label_dir, img_name, dims)

        result = get_sliced_prediction(
            image_path, detection_model,
            slice_height=1024, slice_width=1024,
            overlap_height_ratio=0.2, overlap_width_ratio=0.2,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.3
        )

        preds = [{'bbox': obj.bbox.to_xyxy(), 'category_id': obj.category.id, 'score': obj.score.value}
                 for obj in result.object_prediction_list]

        # Simple counts for Precision/Recall
        matched_gt = [False] * len(gts)
        sorted_p = sorted(preds, key=lambda x: x['score'], reverse=True)
        for p in sorted_p:
            best_iou, best_idx = -1, -1
            for i, g in enumerate(gts):
                if not matched_gt[i] and p['category_id'] == g['category_id']:
                    iou = bbox_iou(p['bbox'], g['bbox'])
                    if iou > best_iou: best_iou, best_idx = iou, i
            if best_iou >= 0.5:
                f_tp += 1
                matched_gt[best_idx] = True
            else: f_fp += 1
        f_fn += matched_gt.count(False)

        # Store for mAP and Small Object metrics
        for p in preds: all_preds_cls[p['category_id']].append(p)
        for g in gts:
            all_gts_cls[g['category_id']].append(g)
            if g['area'] < SMALL_OBJ_THRESHOLD:
                # Local check for small recall
                best_iou = max([bbox_iou(p['bbox'], g['bbox']) for p in preds if p['category_id'] == g['category_id']] + [0])
                if best_iou >= 0.5: f_tp_s += 1
                else: f_fn_s += 1

        if idx < 3: # Save samples
            visual_result = visualize_object_predictions(np.array(read_image(image_path)), result.object_prediction_list)
            cv2.imwrite(os.path.join(OUTPUT_VIS_DIR, f"{folder}_{img_name}"), cv2.cvtColor(visual_result['image'], cv2.COLOR_RGB2BGR))

    # Fold Metrics
    p_f = f_tp / (f_tp + f_fp) if (f_tp + f_fp) > 0 else 0
    r_f = f_tp / (f_tp + f_fn) if (f_tp + f_fn) > 0 else 0
    f1_f = 2*(p_f*r_f)/(p_f+r_f) if (p_f+r_f)>0 else 0
    map50 = np.mean([calculate_ap(all_preds_cls[i], all_gts_cls[i]) for i in range(num_classes) if all_gts_cls[i]])
    s_rec = f_tp_s / (f_tp_s + f_fn_s) if (f_tp_s + f_fn_s) > 0 else 0

    results_log.append({
        'Folder': folder, 'mAP@0.5': round(map50, 4), 'Precision': round(p_f, 4),
        'Recall': round(r_f, 4), 'F1': round(f1_f, 4), 'Small_Recall': round(s_rec, 4)
    })

df = pd.DataFrame(results_log)
avg_row = df.mean(numeric_only=True).to_dict(); avg_row['Folder'] = 'AVERAGE'
df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
print("\n--- Final Results (SAHI + mAP) ---")
print(df.to_string(index=False))


--- 🚀 Starting Evaluation with mAP: fold_0 ---
Performing prediction on 6 slices.
Performing prediction on 35 slices.
Performing prediction on 4 slices.
Performing prediction on 12 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 1 slices.
Performing prediction on 4 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 4 slices.
Performing prediction on 1 slices.
Performing prediction on 4 slices.
Performing prediction on 6 slices.
Performing prediction on 1 slices.
Performing prediction on 6 slices.
Performing prediction on 8 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 6 slices.
Performing prediction on 2 slices.
Performing prediction on 2 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 2 slices.
Performing prediction on 3 slices.
Perfo

###Subset: Non Missing

In [ ]:
import os
import numpy as np
import pandas as pd
import yaml
import time
import cv2
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import visualize_object_predictions, read_image

# --- CONFIGURATION ---
BASE_PATH = '/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data'
OUTPUT_VIS_DIR = '/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV8/non_missing/visuals'
FOLDERS = ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']
SMALL_OBJ_THRESHOLD = 32 * 32
os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)

MODEL_MAP = {
    'fold_0': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/non_missing/fold_0/weights/best.pt',
    'fold_1': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/non_missing/fold_1/weights/best.pt',
    'fold_2': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/non_missing/fold_2/weights/best.pt',
    'fold_3': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/non_missing/fold_3/weights/best.pt',
    'fold_4': '/content/drive/MyDrive/PCB_MC/Results/YOLOV8/non_missing/fold_4/weights/best.pt',
}

# --- HELPER FUNCTIONS ---
def get_image_dimensions(image_path):
    with Image.open(image_path) as img: return img.size

def load_yaml(yaml_path):
    with open(yaml_path, 'r') as f: return yaml.safe_load(f)

def bbox_iou(box1, box2):
    x1, y1, x2, y2 = max(box1[0], box2[0]), max(box1[1], box2[1]), min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def load_yolo_annotations(label_dir, image_filename, image_dims):
    annotations = []
    label_path = os.path.join(label_dir, os.path.splitext(image_filename)[0] + '.txt')
    if not os.path.exists(label_path): return annotations
    img_w, img_h = image_dims
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                c_id, cx, cy, w, h = map(float, parts)
                xmin, ymin = int((cx - w/2) * img_w), int((cy - h/2) * img_h)
                xmax, ymax = int((cx + w/2) * img_w), int((cy + h/2) * img_h)
                annotations.append({'bbox': [xmin, ymin, xmax, ymax], 'category_id': int(c_id), 'area': (xmax-xmin)*(ymax-ymin)})
    return annotations

def calculate_ap(preds, gts, iou_threshold=0.5):
    if not gts: return 0.0
    if not preds: return 0.0
    preds = sorted(preds, key=lambda x: x['score'], reverse=True)
    tp, fp = np.zeros(len(preds)), np.zeros(len(preds))
    matched_gt = [False] * len(gts)
    for i, p in enumerate(preds):
        best_iou, best_idx = -1, -1
        for j, g in enumerate(gts):
            if not matched_gt[j] and p['category_id'] == g['category_id']:
                iou = bbox_iou(p['bbox'], g['bbox'])
                if iou > best_iou: best_iou, best_idx = iou, j
        if best_iou >= iou_threshold:
            tp[i] = 1
            matched_gt[best_idx] = True
        else: fp[i] = 1
    tp_cumsum, fp_cumsum = np.cumsum(tp), np.cumsum(fp)
    recalls = tp_cumsum / len(gts)
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)
    # 11-point interpolation
    ap = 0.0
    for t in np.arange(0, 1.1, 0.1):
        p = np.max(precisions[recalls >= t]) if any(recalls >= t) else 0.0
        ap += p / 11.0
    return ap

# --- MAIN EVALUATION LOOP ---
data_yaml = load_yaml(f"{BASE_PATH}/fold_0/data.yaml")
num_classes = len(data_yaml.get('names', []))
results_log = []

for folder in FOLDERS:
    print(f"\n--- 🚀 Starting Evaluation with mAP: {folder} ---")
    img_dir, label_dir = f"{BASE_PATH}/{folder}/valid/images", f"{BASE_PATH}/{folder}/valid/labels"
    image_filenames = [f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    detection_model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics', model_path=MODEL_MAP[folder],
        confidence_threshold=0.30, device='cuda:0'
    )

    f_tp, f_fp, f_fn = 0, 0, 0
    f_tp_s, f_fn_s = 0, 0
    all_preds_cls = {i: [] for i in range(num_classes)}
    all_gts_cls = {i: [] for i in range(num_classes)}

    for idx, img_name in enumerate(image_filenames):
        image_path = os.path.join(img_dir, img_name)
        dims = get_image_dimensions(image_path)
        gts = load_yolo_annotations(label_dir, img_name, dims)

        result = get_sliced_prediction(
            image_path, detection_model,
            slice_height=1024, slice_width=1024,
            overlap_height_ratio=0.2, overlap_width_ratio=0.2,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.3
        )

        preds = [{'bbox': obj.bbox.to_xyxy(), 'category_id': obj.category.id, 'score': obj.score.value}
                 for obj in result.object_prediction_list]

        # Simple counts for Precision/Recall
        matched_gt = [False] * len(gts)
        sorted_p = sorted(preds, key=lambda x: x['score'], reverse=True)
        for p in sorted_p:
            best_iou, best_idx = -1, -1
            for i, g in enumerate(gts):
                if not matched_gt[i] and p['category_id'] == g['category_id']:
                    iou = bbox_iou(p['bbox'], g['bbox'])
                    if iou > best_iou: best_iou, best_idx = iou, i
            if best_iou >= 0.5:
                f_tp += 1
                matched_gt[best_idx] = True
            else: f_fp += 1
        f_fn += matched_gt.count(False)

        # Store for mAP and Small Object metrics
        for p in preds: all_preds_cls[p['category_id']].append(p)
        for g in gts:
            all_gts_cls[g['category_id']].append(g)
            if g['area'] < SMALL_OBJ_THRESHOLD:
                # Local check for small recall
                best_iou = max([bbox_iou(p['bbox'], g['bbox']) for p in preds if p['category_id'] == g['category_id']] + [0])
                if best_iou >= 0.5: f_tp_s += 1
                else: f_fn_s += 1

        if idx < 3: # Save samples
            visual_result = visualize_object_predictions(np.array(read_image(image_path)), result.object_prediction_list)
            cv2.imwrite(os.path.join(OUTPUT_VIS_DIR, f"{folder}_{img_name}"), cv2.cvtColor(visual_result['image'], cv2.COLOR_RGB2BGR))

    # Fold Metrics
    p_f = f_tp / (f_tp + f_fp) if (f_tp + f_fp) > 0 else 0
    r_f = f_tp / (f_tp + f_fn) if (f_tp + f_fn) > 0 else 0
    f1_f = 2*(p_f*r_f)/(p_f+r_f) if (p_f+r_f)>0 else 0
    map50 = np.mean([calculate_ap(all_preds_cls[i], all_gts_cls[i]) for i in range(num_classes) if all_gts_cls[i]])
    s_rec = f_tp_s / (f_tp_s + f_fn_s) if (f_tp_s + f_fn_s) > 0 else 0

    results_log.append({
        'Folder': folder, 'mAP@0.5': round(map50, 4), 'Precision': round(p_f, 4),
        'Recall': round(r_f, 4), 'F1': round(f1_f, 4), 'Small_Recall': round(s_rec, 4)
    })

df = pd.DataFrame(results_log)
avg_row = df.mean(numeric_only=True).to_dict(); avg_row['Folder'] = 'AVERAGE'
df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
print("\n--- Final Results (SAHI + mAP) ---")
print(df.to_string(index=False))


--- 🚀 Starting Evaluation with mAP: fold_0 ---
Performing prediction on 3 slices.
Performing prediction on 3 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 4 slices.
Performing prediction on 1 slices.
Performing prediction on 6 slices.
Performing prediction on 20 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 8 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 3 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 4 slices.
Performing prediction on 20 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 9 slices.
Perfo

##Visualizations

###Subset: Full Dataset + Visualization

In [ ]:
import os
import cv2
import yaml
import numpy as np
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# ============================================================
# CONFIG
# ============================================================

BASE_PATH = "/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data"
DATA_YAML = "/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_0/data.yaml"

OUT_ROOT = "/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV8/full_dataset/tp_fp_fn_split"

FOLDS = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]

MODEL_MAP = {
    "fold_0": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/full_dataset/fold_0/weights/best.pt",
    "fold_1": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/full_dataset/fold_1/weights/best.pt",
    "fold_2": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/full_dataset/fold_2/weights/best.pt",
    "fold_3": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/full_dataset/fold_3/weights/best.pt",
    "fold_4": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/full_dataset/fold_4/weights/best.pt",
}

os.makedirs(OUT_ROOT, exist_ok=True)

# SAHI
SLICE = 640
OVERLAP = 0.2

CONF_MODEL = 0.30
CONF_LOW = 0.80
IOU_TH = 0.5

SAVE_N = 50  # images per fold

# ============================================================
# COLORS (BGR)
# ============================================================

MAGENTA = (255, 0, 255)
YELLOW  = (0, 255, 255)
RED     = (0, 0, 255)
ORANGE  = (0, 165, 255)
WHITE   = (255, 255, 255)
BLACK   = (0, 0, 0)

# ============================================================
# LOAD CLASS INFO
# ============================================================

with open(DATA_YAML, "r") as f:
    data_yaml = yaml.safe_load(f)

ID2NAME = {i: n for i, n in enumerate(data_yaml["names"])}

CONNECTOR_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "connector" in n.lower()
}

MISSING_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "missing" in n.lower()
}

print("Connector IDs:", CONNECTOR_CLASS_IDS)
print("Missing IDs:", MISSING_CLASS_IDS)

# ============================================================
# HELPERS
# ============================================================

def get_image_dimensions(path):
    with Image.open(path) as im:
        return im.size


def draw_text(img, text, x, y):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, BLACK, 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, WHITE, 1, cv2.LINE_AA)


def bbox_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter == 0:
        return 0.0

    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])

    return inter / (areaA + areaB - inter + 1e-9)


def load_gt(label_dir, img_name, dims):
    gts = []

    path = os.path.join(
        label_dir,
        os.path.splitext(img_name)[0] + ".txt"
    )

    if not os.path.exists(path):
        return gts

    W, H = dims

    with open(path) as f:
        for line in f:
            c, cx, cy, w, h = map(float, line.split())

            x1 = (cx - w/2) * W
            y1 = (cy - h/2) * H
            x2 = (cx + w/2) * W
            y2 = (cy + h/2) * H

            gts.append({
                "category_id": int(c),
                "bbox": [x1, y1, x2, y2]
            })

    return gts


def match(preds, gts):

    preds = sorted(preds, key=lambda x: x["score"], reverse=True)
    used = [False]*len(gts)

    tp, fp = [], []

    for p in preds:

        best_iou = 0
        best_j = -1

        for j, g in enumerate(gts):

            if used[j]:
                continue

            if p["category_id"] != g["category_id"]:
                continue

            iou = bbox_iou(p["bbox"], g["bbox"])

            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= IOU_TH:
            used[best_j] = True
            tp.append(p)
        else:
            fp.append(p)

    fn = [gts[i] for i in range(len(gts)) if not used[i]]

    return tp, fp, fn


def class_color(cid, mode):

    if mode == "fp":
        return ORANGE

    if mode == "fn":
        return RED

    if cid in CONNECTOR_CLASS_IDS:
        return YELLOW

    return MAGENTA


def should_label(cid, score, mode):

    if cid in MISSING_CLASS_IDS:
        return True

    if mode in ["fp", "fn"]:
        return True

    if score is not None and score < CONF_LOW:
        return True

    return False


def draw_boxes(img, items, mode):

    for it in items:

        cid = it["category_id"]
        score = it.get("score")

        x1, y1, x2, y2 = map(int, it["bbox"])

        color = class_color(cid, mode)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        if should_label(cid, score, mode):

            name = ID2NAME[cid]

            if mode == "fp":
                text = f"FP {name}:{score:.2f}"
            elif mode == "fn":
                text = f"FN {name}"
            else:
                text = f"{name}:{score:.2f}"

            draw_text(img, text, x1, max(15, y1-5))


def save_views(img_path, tp, fp, fn, out_base):

    img = cv2.imread(img_path)

    tp_img = img.copy()
    fp_img = img.copy()
    fn_img = img.copy()

    draw_boxes(tp_img, tp, "tp")
    draw_boxes(fp_img, fp, "fp")
    draw_boxes(fn_img, fn, "fn")

    cv2.imwrite(out_base + "_TP.png", tp_img)
    cv2.imwrite(out_base + "_FP.png", fp_img)
    cv2.imwrite(out_base + "_FN.png", fn_img)


# ============================================================
# MAIN LOOP
# ============================================================

for fold in FOLDS:

    print(f"\n🚀 Processing {fold}")

    img_dir = f"{BASE_PATH}/{fold}/valid/images"
    label_dir = f"{BASE_PATH}/{fold}/valid/labels"

    out_dir = os.path.join(OUT_ROOT, fold)
    os.makedirs(out_dir, exist_ok=True)

    model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=MODEL_MAP[fold],
        confidence_threshold=CONF_MODEL,
        device="cuda:0"
    )

    images = sorted(os.listdir(img_dir))[:SAVE_N]

    for img_name in images:

        img_path = os.path.join(img_dir, img_name)

        dims = get_image_dimensions(img_path)
        gts = load_gt(label_dir, img_name, dims)

        result = get_sliced_prediction(
            img_path,
            model,
            slice_height=SLICE,
            slice_width=SLICE,
            overlap_height_ratio=OVERLAP,
            overlap_width_ratio=OVERLAP,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.5
        )

        preds = []

        for obj in result.object_prediction_list:

            x1, y1, x2, y2 = obj.bbox.to_xyxy()

            preds.append({
                "category_id": int(obj.category.id),
                "score": float(obj.score.value),
                "bbox": [x1, y1, x2, y2]
            })

        tp, fp, fn = match(preds, gts)

        out_base = os.path.join(
            out_dir,
            os.path.splitext(img_name)[0]
        )

        save_views(img_path, tp, fp, fn, out_base)

    print(f"✅ Saved to {out_dir}")

print("\nDONE ✅")


Connector IDs: {3}
Missing IDs: {23, 24, 25, 26, 27, 28, 29, 30}

🚀 Processing fold_0
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Performing prediction on 2 slices.
Performing prediction on 2 slices.
Performing prediction on 12 slices.
Performing prediction on 40 slices.
Performing prediction on 30 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 12 slices.
Performing prediction on 2 slices.
Performing prediction on 12 slices.
Performing prediction on 8 slices.
Performing prediction on 12 slices.
Performing prediction on 8 slices.
Performing prediction on 8 slices.
Performing prediction on 40 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.

###Subset: Components Only  + Visualization

In [ ]:
import os
import cv2
import yaml
import numpy as np
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# ============================================================
# CONFIG
# ============================================================

BASE_PATH = "/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data"
DATA_YAML = "/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_0/data.yaml"

OUT_ROOT = "/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV8/components_only/tp_fp_fn_split"

FOLDS = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]

MODEL_MAP = {
    "fold_0": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/components_only/fold_0/weights/best.pt",
    "fold_1": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/components_only/fold_1/weights/best.pt",
    "fold_2": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/components_only/fold_2/weights/best.pt",
    "fold_3": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/components_only/fold_3/weights/best.pt",
    "fold_4": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/components_only/fold_4/weights/best.pt",
}

os.makedirs(OUT_ROOT, exist_ok=True)

# SAHI
SLICE = 640
OVERLAP = 0.2

CONF_MODEL = 0.30
CONF_LOW = 0.80
IOU_TH = 0.5

SAVE_N = 50  # images per fold

# ============================================================
# COLORS (BGR)
# ============================================================

MAGENTA = (255, 0, 255)
YELLOW  = (0, 255, 255)
RED     = (0, 0, 255)
ORANGE  = (0, 165, 255)
WHITE   = (255, 255, 255)
BLACK   = (0, 0, 0)

# ============================================================
# LOAD CLASS INFO
# ============================================================

with open(DATA_YAML, "r") as f:
    data_yaml = yaml.safe_load(f)

ID2NAME = {i: n for i, n in enumerate(data_yaml["names"])}

CONNECTOR_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "connector" in n.lower()
}

MISSING_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "missing" in n.lower()
}

print("Connector IDs:", CONNECTOR_CLASS_IDS)
print("Missing IDs:", MISSING_CLASS_IDS)

# ============================================================
# HELPERS
# ============================================================

def get_image_dimensions(path):
    with Image.open(path) as im:
        return im.size


def draw_text(img, text, x, y):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, BLACK, 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, WHITE, 1, cv2.LINE_AA)


def bbox_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter == 0:
        return 0.0

    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])

    return inter / (areaA + areaB - inter + 1e-9)


def load_gt(label_dir, img_name, dims):
    gts = []

    path = os.path.join(
        label_dir,
        os.path.splitext(img_name)[0] + ".txt"
    )

    if not os.path.exists(path):
        return gts

    W, H = dims

    with open(path) as f:
        for line in f:
            c, cx, cy, w, h = map(float, line.split())

            x1 = (cx - w/2) * W
            y1 = (cy - h/2) * H
            x2 = (cx + w/2) * W
            y2 = (cy + h/2) * H

            gts.append({
                "category_id": int(c),
                "bbox": [x1, y1, x2, y2]
            })

    return gts


def match(preds, gts):

    preds = sorted(preds, key=lambda x: x["score"], reverse=True)
    used = [False]*len(gts)

    tp, fp = [], []

    for p in preds:

        best_iou = 0
        best_j = -1

        for j, g in enumerate(gts):

            if used[j]:
                continue

            if p["category_id"] != g["category_id"]:
                continue

            iou = bbox_iou(p["bbox"], g["bbox"])

            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= IOU_TH:
            used[best_j] = True
            tp.append(p)
        else:
            fp.append(p)

    fn = [gts[i] for i in range(len(gts)) if not used[i]]

    return tp, fp, fn


def class_color(cid, mode):

    if mode == "fp":
        return ORANGE

    if mode == "fn":
        return RED

    if cid in CONNECTOR_CLASS_IDS:
        return YELLOW

    return MAGENTA


def should_label(cid, score, mode):

    if cid in MISSING_CLASS_IDS:
        return True

    if mode in ["fp", "fn"]:
        return True

    if score is not None and score < CONF_LOW:
        return True

    return False


def draw_boxes(img, items, mode):

    for it in items:

        cid = it["category_id"]
        score = it.get("score")

        x1, y1, x2, y2 = map(int, it["bbox"])

        color = class_color(cid, mode)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        if should_label(cid, score, mode):

            name = ID2NAME[cid]

            if mode == "fp":
                text = f"FP {name}:{score:.2f}"
            elif mode == "fn":
                text = f"FN {name}"
            else:
                text = f"{name}:{score:.2f}"

            draw_text(img, text, x1, max(15, y1-5))


def save_views(img_path, tp, fp, fn, out_base):

    img = cv2.imread(img_path)

    tp_img = img.copy()
    fp_img = img.copy()
    fn_img = img.copy()

    draw_boxes(tp_img, tp, "tp")
    draw_boxes(fp_img, fp, "fp")
    draw_boxes(fn_img, fn, "fn")

    cv2.imwrite(out_base + "_TP.png", tp_img)
    cv2.imwrite(out_base + "_FP.png", fp_img)
    cv2.imwrite(out_base + "_FN.png", fn_img)


# ============================================================
# MAIN LOOP
# ============================================================

for fold in FOLDS:

    print(f"\n🚀 Processing {fold}")

    img_dir = f"{BASE_PATH}/{fold}/valid/images"
    label_dir = f"{BASE_PATH}/{fold}/valid/labels"

    out_dir = os.path.join(OUT_ROOT, fold)
    os.makedirs(out_dir, exist_ok=True)

    model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=MODEL_MAP[fold],
        confidence_threshold=CONF_MODEL,
        device="cuda:0"
    )

    images = sorted(os.listdir(img_dir))[:SAVE_N]

    for img_name in images:

        img_path = os.path.join(img_dir, img_name)

        dims = get_image_dimensions(img_path)
        gts = load_gt(label_dir, img_name, dims)

        result = get_sliced_prediction(
            img_path,
            model,
            slice_height=SLICE,
            slice_width=SLICE,
            overlap_height_ratio=OVERLAP,
            overlap_width_ratio=OVERLAP,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.5
        )

        preds = []

        for obj in result.object_prediction_list:

            x1, y1, x2, y2 = obj.bbox.to_xyxy()

            preds.append({
                "category_id": int(obj.category.id),
                "score": float(obj.score.value),
                "bbox": [x1, y1, x2, y2]
            })

        tp, fp, fn = match(preds, gts)

        out_base = os.path.join(
            out_dir,
            os.path.splitext(img_name)[0]
        )

        save_views(img_path, tp, fp, fn, out_base)

    print(f"✅ Saved to {out_dir}")

print("\nDONE ✅")


Connector IDs: {3}
Missing IDs: set()

🚀 Processing fold_0
Performing prediction on 2 slices.
Performing prediction on 2 slices.
Performing prediction on 40 slices.
Performing prediction on 12 slices.
Performing prediction on 30 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 48 slices.
Performing prediction on 2 slices.
Performing prediction on 12 slices.
Performing prediction on 40 slices.
Performing prediction on 48 slices.
Performing prediction on 8 slices.
Performing prediction on 8 slices.
Performing prediction on 12 slices.
Performing prediction on 40 slices.
Performing prediction on 40 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 1 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 48 slices.
Performing pr

###Subset: Missing Only  + Visualization

In [ ]:
import os
import cv2
import yaml
import numpy as np
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# ============================================================
# CONFIG
# ============================================================

BASE_PATH = "/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data"
DATA_YAML = "/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/data.yaml"

OUT_ROOT = "/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV8/missing_only/tp_fp_fn_split"

FOLDS = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]

MODEL_MAP = {
    "fold_0": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/missing_only/fold_0/weights/best.pt",
    "fold_1": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/missing_only/fold_1/weights/best.pt",
    "fold_2": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/missing_only/fold_2/weights/best.pt",
    "fold_3": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/missing_only/fold_3/weights/best.pt",
    "fold_4": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/missing_only/fold_4/weights/best.pt",
}

os.makedirs(OUT_ROOT, exist_ok=True)

# SAHI
SLICE = 640
OVERLAP = 0.2

CONF_MODEL = 0.30
CONF_LOW = 0.80
IOU_TH = 0.5

SAVE_N = 50  # images per fold

# ============================================================
# COLORS (BGR)
# ============================================================

MAGENTA = (255, 0, 255)
YELLOW  = (0, 255, 255)
RED     = (0, 0, 255)
ORANGE  = (0, 165, 255)
WHITE   = (255, 255, 255)
BLACK   = (0, 0, 0)

# ============================================================
# LOAD CLASS INFO
# ============================================================

with open(DATA_YAML, "r") as f:
    data_yaml = yaml.safe_load(f)

ID2NAME = {i: n for i, n in enumerate(data_yaml["names"])}

CONNECTOR_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "connector" in n.lower()
}

MISSING_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "missing" in n.lower()
}

print("Connector IDs:", CONNECTOR_CLASS_IDS)
print("Missing IDs:", MISSING_CLASS_IDS)

# ============================================================
# HELPERS
# ============================================================

def get_image_dimensions(path):
    with Image.open(path) as im:
        return im.size


def draw_text(img, text, x, y):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, BLACK, 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, WHITE, 1, cv2.LINE_AA)


def bbox_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter == 0:
        return 0.0

    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])

    return inter / (areaA + areaB - inter + 1e-9)


def load_gt(label_dir, img_name, dims):
    gts = []

    path = os.path.join(
        label_dir,
        os.path.splitext(img_name)[0] + ".txt"
    )

    if not os.path.exists(path):
        return gts

    W, H = dims

    with open(path) as f:
        for line in f:
            c, cx, cy, w, h = map(float, line.split())

            x1 = (cx - w/2) * W
            y1 = (cy - h/2) * H
            x2 = (cx + w/2) * W
            y2 = (cy + h/2) * H

            gts.append({
                "category_id": int(c),
                "bbox": [x1, y1, x2, y2]
            })

    return gts


def match(preds, gts):

    preds = sorted(preds, key=lambda x: x["score"], reverse=True)
    used = [False]*len(gts)

    tp, fp = [], []

    for p in preds:

        best_iou = 0
        best_j = -1

        for j, g in enumerate(gts):

            if used[j]:
                continue

            if p["category_id"] != g["category_id"]:
                continue

            iou = bbox_iou(p["bbox"], g["bbox"])

            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= IOU_TH:
            used[best_j] = True
            tp.append(p)
        else:
            fp.append(p)

    fn = [gts[i] for i in range(len(gts)) if not used[i]]

    return tp, fp, fn


def class_color(cid, mode):

    if mode == "fp":
        return ORANGE

    if mode == "fn":
        return RED

    if cid in CONNECTOR_CLASS_IDS:
        return YELLOW

    return MAGENTA


def should_label(cid, score, mode):

    if cid in MISSING_CLASS_IDS:
        return True

    if mode in ["fp", "fn"]:
        return True

    if score is not None and score < CONF_LOW:
        return True

    return False


def draw_boxes(img, items, mode):

    for it in items:

        cid = it["category_id"]
        score = it.get("score")

        x1, y1, x2, y2 = map(int, it["bbox"])

        color = class_color(cid, mode)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        if should_label(cid, score, mode):

            name = ID2NAME[cid]

            if mode == "fp":
                text = f"FP {name}:{score:.2f}"
            elif mode == "fn":
                text = f"FN {name}"
            else:
                text = f"{name}:{score:.2f}"

            draw_text(img, text, x1, max(15, y1-5))


def save_views(img_path, tp, fp, fn, out_base):

    img = cv2.imread(img_path)

    tp_img = img.copy()
    fp_img = img.copy()
    fn_img = img.copy()

    draw_boxes(tp_img, tp, "tp")
    draw_boxes(fp_img, fp, "fp")
    draw_boxes(fn_img, fn, "fn")

    cv2.imwrite(out_base + "_TP.png", tp_img)
    cv2.imwrite(out_base + "_FP.png", fp_img)
    cv2.imwrite(out_base + "_FN.png", fn_img)


# ============================================================
# MAIN LOOP
# ============================================================

for fold in FOLDS:

    print(f"\n🚀 Processing {fold}")

    img_dir = f"{BASE_PATH}/{fold}/valid/images"
    label_dir = f"{BASE_PATH}/{fold}/valid/labels"

    out_dir = os.path.join(OUT_ROOT, fold)
    os.makedirs(out_dir, exist_ok=True)

    model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=MODEL_MAP[fold],
        confidence_threshold=CONF_MODEL,
        device="cuda:0"
    )

    images = sorted(os.listdir(img_dir))[:SAVE_N]

    for img_name in images:

        img_path = os.path.join(img_dir, img_name)

        dims = get_image_dimensions(img_path)
        gts = load_gt(label_dir, img_name, dims)

        result = get_sliced_prediction(
            img_path,
            model,
            slice_height=SLICE,
            slice_width=SLICE,
            overlap_height_ratio=OVERLAP,
            overlap_width_ratio=OVERLAP,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.5
        )

        preds = []

        for obj in result.object_prediction_list:

            x1, y1, x2, y2 = obj.bbox.to_xyxy()

            preds.append({
                "category_id": int(obj.category.id),
                "score": float(obj.score.value),
                "bbox": [x1, y1, x2, y2]
            })

        tp, fp, fn = match(preds, gts)

        out_base = os.path.join(
            out_dir,
            os.path.splitext(img_name)[0]
        )

        save_views(img_path, tp, fp, fn, out_base)

    print(f"✅ Saved to {out_dir}")

print("\nDONE ✅")


Connector IDs: set()
Missing IDs: {0, 1, 2, 3, 4, 5, 6, 7}

🚀 Processing fold_0
Performing prediction on 2 slices.
Performing prediction on 15 slices.
Performing prediction on 40 slices.
Performing prediction on 9 slices.
Performing prediction on 8 slices.
Performing prediction on 8 slices.
Performing prediction on 48 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 6 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 8 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 24 slices.
Performing prediction on 12 slices.
Performing prediction on 6 slices.
Performing prediction on 4 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 8 slices.
Performing prediction on 8 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slic

###Subset: Non Missing  + Visualization

In [ ]:
import os
import cv2
import yaml
import numpy as np
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# ============================================================
# CONFIG
# ============================================================

BASE_PATH = "/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data"
DATA_YAML = "/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_0/data.yaml"

OUT_ROOT = "/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV8/non_missing/tp_fp_fn_split"

FOLDS = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]

MODEL_MAP = {
    "fold_0": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/non_missing/fold_0/weights/best.pt",
    "fold_1": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/non_missing/fold_1/weights/best.pt",
    "fold_2": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/non_missing/fold_2/weights/best.pt",
    "fold_3": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/non_missing/fold_3/weights/best.pt",
    "fold_4": "/content/drive/MyDrive/PCB_MC/Results/YOLOV8/non_missing/fold_4/weights/best.pt",
}

os.makedirs(OUT_ROOT, exist_ok=True)

# SAHI
SLICE = 640
OVERLAP = 0.2

CONF_MODEL = 0.30
CONF_LOW = 0.80
IOU_TH = 0.5

SAVE_N = 50  # images per fold

# ============================================================
# COLORS (BGR)
# ============================================================

MAGENTA = (255, 0, 255)
YELLOW  = (0, 255, 255)
RED     = (0, 0, 255)
ORANGE  = (0, 165, 255)
WHITE   = (255, 255, 255)
BLACK   = (0, 0, 0)

# ============================================================
# LOAD CLASS INFO
# ============================================================

with open(DATA_YAML, "r") as f:
    data_yaml = yaml.safe_load(f)

ID2NAME = {i: n for i, n in enumerate(data_yaml["names"])}

CONNECTOR_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "connector" in n.lower()
}

MISSING_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "missing" in n.lower()
}

print("Connector IDs:", CONNECTOR_CLASS_IDS)
print("Missing IDs:", MISSING_CLASS_IDS)

# ============================================================
# HELPERS
# ============================================================

def get_image_dimensions(path):
    with Image.open(path) as im:
        return im.size


def draw_text(img, text, x, y):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, BLACK, 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, WHITE, 1, cv2.LINE_AA)


def bbox_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter == 0:
        return 0.0

    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])

    return inter / (areaA + areaB - inter + 1e-9)


def load_gt(label_dir, img_name, dims):
    gts = []

    path = os.path.join(
        label_dir,
        os.path.splitext(img_name)[0] + ".txt"
    )

    if not os.path.exists(path):
        return gts

    W, H = dims

    with open(path) as f:
        for line in f:
            c, cx, cy, w, h = map(float, line.split())

            x1 = (cx - w/2) * W
            y1 = (cy - h/2) * H
            x2 = (cx + w/2) * W
            y2 = (cy + h/2) * H

            gts.append({
                "category_id": int(c),
                "bbox": [x1, y1, x2, y2]
            })

    return gts


def match(preds, gts):

    preds = sorted(preds, key=lambda x: x["score"], reverse=True)
    used = [False]*len(gts)

    tp, fp = [], []

    for p in preds:

        best_iou = 0
        best_j = -1

        for j, g in enumerate(gts):

            if used[j]:
                continue

            if p["category_id"] != g["category_id"]:
                continue

            iou = bbox_iou(p["bbox"], g["bbox"])

            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= IOU_TH:
            used[best_j] = True
            tp.append(p)
        else:
            fp.append(p)

    fn = [gts[i] for i in range(len(gts)) if not used[i]]

    return tp, fp, fn


def class_color(cid, mode):

    if mode == "fp":
        return ORANGE

    if mode == "fn":
        return RED

    if cid in CONNECTOR_CLASS_IDS:
        return YELLOW

    return MAGENTA


def should_label(cid, score, mode):

    if cid in MISSING_CLASS_IDS:
        return True

    if mode in ["fp", "fn"]:
        return True

    if score is not None and score < CONF_LOW:
        return True

    return False


def draw_boxes(img, items, mode):

    for it in items:

        cid = it["category_id"]
        score = it.get("score")

        x1, y1, x2, y2 = map(int, it["bbox"])

        color = class_color(cid, mode)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        if should_label(cid, score, mode):

            name = ID2NAME[cid]

            if mode == "fp":
                text = f"FP {name}:{score:.2f}"
            elif mode == "fn":
                text = f"FN {name}"
            else:
                text = f"{name}:{score:.2f}"

            draw_text(img, text, x1, max(15, y1-5))


def save_views(img_path, tp, fp, fn, out_base):

    img = cv2.imread(img_path)

    tp_img = img.copy()
    fp_img = img.copy()
    fn_img = img.copy()

    draw_boxes(tp_img, tp, "tp")
    draw_boxes(fp_img, fp, "fp")
    draw_boxes(fn_img, fn, "fn")

    cv2.imwrite(out_base + "_TP.png", tp_img)
    cv2.imwrite(out_base + "_FP.png", fp_img)
    cv2.imwrite(out_base + "_FN.png", fn_img)


# ============================================================
# MAIN LOOP
# ============================================================

for fold in FOLDS:

    print(f"\n🚀 Processing {fold}")

    img_dir = f"{BASE_PATH}/{fold}/valid/images"
    label_dir = f"{BASE_PATH}/{fold}/valid/labels"

    out_dir = os.path.join(OUT_ROOT, fold)
    os.makedirs(out_dir, exist_ok=True)

    model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=MODEL_MAP[fold],
        confidence_threshold=CONF_MODEL,
        device="cuda:0"
    )

    images = sorted(os.listdir(img_dir))[:SAVE_N]

    for img_name in images:

        img_path = os.path.join(img_dir, img_name)

        dims = get_image_dimensions(img_path)
        gts = load_gt(label_dir, img_name, dims)

        result = get_sliced_prediction(
            img_path,
            model,
            slice_height=SLICE,
            slice_width=SLICE,
            overlap_height_ratio=OVERLAP,
            overlap_width_ratio=OVERLAP,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.5
        )

        preds = []

        for obj in result.object_prediction_list:

            x1, y1, x2, y2 = obj.bbox.to_xyxy()

            preds.append({
                "category_id": int(obj.category.id),
                "score": float(obj.score.value),
                "bbox": [x1, y1, x2, y2]
            })

        tp, fp, fn = match(preds, gts)

        out_base = os.path.join(
            out_dir,
            os.path.splitext(img_name)[0]
        )

        save_views(img_path, tp, fp, fn, out_base)

    print(f"✅ Saved to {out_dir}")

print("\nDONE ✅")


Connector IDs: {3}
Missing IDs: set()

🚀 Processing fold_0
Performing prediction on 48 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 48 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 40 slices.
Performing prediction on 12 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 32 slices.
Performing prediction on 2 slices.
Performing prediction on 16 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 96 slices.
Performing prediction on 4 slices.
Performing prediction on 3 slices.
Performing prediction on 12 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 16 slices.
Performing prediction on 20 slices.
Performing p

ValueError: too many values to unpack (expected 5)

# YOLO V11 + SAHI

## Training

###Subset: Full Dataset

In [4]:
import os
import numpy as np
import pandas as pd
import yaml
import time
import cv2
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import visualize_object_predictions, read_image

# --- CONFIGURATION ---
BASE_PATH = '/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data'
OUTPUT_VIS_DIR = '/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV11/full_dataset/visuals'
FOLDERS = ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']
SMALL_OBJ_THRESHOLD = 32 * 32
os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)

MODEL_MAP = {
    'fold_0': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/full_dataset/fold_0/weights/best.pt',
    'fold_1': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/full_dataset/fold_1/weights/best.pt',
    'fold_2': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/full_dataset/fold_2/weights/best.pt',
    'fold_3': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/full_dataset/fold_3/weights/best.pt',
    'fold_4': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/full_dataset/fold_4/weights/best.pt',
}

# --- HELPER FUNCTIONS ---
def get_image_dimensions(image_path):
    with Image.open(image_path) as img: return img.size

def load_yaml(yaml_path):
    with open(yaml_path, 'r') as f: return yaml.safe_load(f)

def bbox_iou(box1, box2):
    x1, y1, x2, y2 = max(box1[0], box2[0]), max(box1[1], box2[1]), min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def load_yolo_annotations(label_dir, image_filename, image_dims):
    annotations = []
    label_path = os.path.join(label_dir, os.path.splitext(image_filename)[0] + '.txt')
    if not os.path.exists(label_path): return annotations
    img_w, img_h = image_dims
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                c_id, cx, cy, w, h = map(float, parts)
                xmin, ymin = int((cx - w/2) * img_w), int((cy - h/2) * img_h)
                xmax, ymax = int((cx + w/2) * img_w), int((cy + h/2) * img_h)
                annotations.append({'bbox': [xmin, ymin, xmax, ymax], 'category_id': int(c_id), 'area': (xmax-xmin)*(ymax-ymin)})
    return annotations

def calculate_ap(preds, gts, iou_threshold=0.5):
    if not gts: return 0.0
    if not preds: return 0.0
    preds = sorted(preds, key=lambda x: x['score'], reverse=True)
    tp, fp = np.zeros(len(preds)), np.zeros(len(preds))
    matched_gt = [False] * len(gts)
    for i, p in enumerate(preds):
        best_iou, best_idx = -1, -1
        for j, g in enumerate(gts):
            if not matched_gt[j] and p['category_id'] == g['category_id']:
                iou = bbox_iou(p['bbox'], g['bbox'])
                if iou > best_iou: best_iou, best_idx = iou, j
        if best_iou >= iou_threshold:
            tp[i] = 1
            matched_gt[best_idx] = True
        else: fp[i] = 1
    tp_cumsum, fp_cumsum = np.cumsum(tp), np.cumsum(fp)
    recalls = tp_cumsum / len(gts)
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)
    # 11-point interpolation
    ap = 0.0
    for t in np.arange(0, 1.1, 0.1):
        p = np.max(precisions[recalls >= t]) if any(recalls >= t) else 0.0
        ap += p / 11.0
    return ap

# --- MAIN EVALUATION LOOP ---
data_yaml = load_yaml(f"{BASE_PATH}/fold_0/data.yaml")
num_classes = len(data_yaml.get('names', []))
results_log = []

for folder in FOLDERS:
    print(f"\n--- 🚀 Starting Evaluation with mAP: {folder} ---")
    img_dir, label_dir = f"{BASE_PATH}/{folder}/valid/images", f"{BASE_PATH}/{folder}/valid/labels"
    image_filenames = [f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    detection_model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics', model_path=MODEL_MAP[folder],
        confidence_threshold=0.30, device='cuda:0'
    )

    f_tp, f_fp, f_fn = 0, 0, 0
    f_tp_s, f_fn_s = 0, 0
    all_preds_cls = {i: [] for i in range(num_classes)}
    all_gts_cls = {i: [] for i in range(num_classes)}

    for idx, img_name in enumerate(image_filenames):
        image_path = os.path.join(img_dir, img_name)
        dims = get_image_dimensions(image_path)
        gts = load_yolo_annotations(label_dir, img_name, dims)

        result = get_sliced_prediction(
            image_path, detection_model,
            slice_height=1024, slice_width=1024,
            overlap_height_ratio=0.2, overlap_width_ratio=0.2,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.3
        )

        preds = [{'bbox': obj.bbox.to_xyxy(), 'category_id': obj.category.id, 'score': obj.score.value}
                 for obj in result.object_prediction_list]

        # Simple counts for Precision/Recall
        matched_gt = [False] * len(gts)
        sorted_p = sorted(preds, key=lambda x: x['score'], reverse=True)
        for p in sorted_p:
            best_iou, best_idx = -1, -1
            for i, g in enumerate(gts):
                if not matched_gt[i] and p['category_id'] == g['category_id']:
                    iou = bbox_iou(p['bbox'], g['bbox'])
                    if iou > best_iou: best_iou, best_idx = iou, i
            if best_iou >= 0.5:
                f_tp += 1
                matched_gt[best_idx] = True
            else: f_fp += 1
        f_fn += matched_gt.count(False)

        # Store for mAP and Small Object metrics
        for p in preds: all_preds_cls[p['category_id']].append(p)
        for g in gts:
            all_gts_cls[g['category_id']].append(g)
            if g['area'] < SMALL_OBJ_THRESHOLD:
                # Local check for small recall
                best_iou = max([bbox_iou(p['bbox'], g['bbox']) for p in preds if p['category_id'] == g['category_id']] + [0])
                if best_iou >= 0.5: f_tp_s += 1
                else: f_fn_s += 1

        if idx < 3: # Save samples
            visual_result = visualize_object_predictions(np.array(read_image(image_path)), result.object_prediction_list)
            cv2.imwrite(os.path.join(OUTPUT_VIS_DIR, f"{folder}_{img_name}"), cv2.cvtColor(visual_result['image'], cv2.COLOR_RGB2BGR))

    # Fold Metrics
    p_f = f_tp / (f_tp + f_fp) if (f_tp + f_fp) > 0 else 0
    r_f = f_tp / (f_tp + f_fn) if (f_tp + f_fn) > 0 else 0
    f1_f = 2*(p_f*r_f)/(p_f+r_f) if (p_f+r_f)>0 else 0
    map50 = np.mean([calculate_ap(all_preds_cls[i], all_gts_cls[i]) for i in range(num_classes) if all_gts_cls[i]])
    s_rec = f_tp_s / (f_tp_s + f_fn_s) if (f_tp_s + f_fn_s) > 0 else 0

    results_log.append({
        'Folder': folder, 'mAP@0.5': round(map50, 4), 'Precision': round(p_f, 4),
        'Recall': round(r_f, 4), 'F1': round(f1_f, 4), 'Small_Recall': round(s_rec, 4)
    })

df = pd.DataFrame(results_log)
avg_row = df.mean(numeric_only=True).to_dict(); avg_row['Folder'] = 'AVERAGE'
df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
print("\n--- Final Results (SAHI + mAP) ---")
print(df.to_string(index=False))


--- 🚀 Starting Evaluation with mAP: fold_0 ---
Performing prediction on 9 slices.
Performing prediction on 40 slices.
Performing prediction on 6 slices.
Performing prediction on 3 slices.
Performing prediction on 3 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 9 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 20 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 4 slices.
Performing prediction on 6 slices.
Performing prediction on 20 slices.
Performing prediction on 35 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 20 slices.
Performing prediction on 6 slices.
Pe

###Subset: Components Only

In [3]:
import os
import numpy as np
import pandas as pd
import yaml
import time
import cv2
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import visualize_object_predictions, read_image

# --- CONFIGURATION ---
BASE_PATH = '/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data'
OUTPUT_VIS_DIR = '/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV11/components_only/visuals'
FOLDERS = ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']
SMALL_OBJ_THRESHOLD = 32 * 32
os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)

MODEL_MAP = {
    'fold_0': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/components_only/fold_0/weights/best.pt',
    'fold_1': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/components_only/fold_1/weights/best.pt',
    'fold_2': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/components_only/fold_2/weights/best.pt',
    'fold_3': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/components_only/fold_3/weights/best.pt',
    'fold_4': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/components_only/fold_4/weights/best.pt',
}

# --- HELPER FUNCTIONS ---
def get_image_dimensions(image_path):
    with Image.open(image_path) as img: return img.size

def load_yaml(yaml_path):
    with open(yaml_path, 'r') as f: return yaml.safe_load(f)

def bbox_iou(box1, box2):
    x1, y1, x2, y2 = max(box1[0], box2[0]), max(box1[1], box2[1]), min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def load_yolo_annotations(label_dir, image_filename, image_dims):
    annotations = []
    label_path = os.path.join(label_dir, os.path.splitext(image_filename)[0] + '.txt')
    if not os.path.exists(label_path): return annotations
    img_w, img_h = image_dims
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                c_id, cx, cy, w, h = map(float, parts)
                xmin, ymin = int((cx - w/2) * img_w), int((cy - h/2) * img_h)
                xmax, ymax = int((cx + w/2) * img_w), int((cy + h/2) * img_h)
                annotations.append({'bbox': [xmin, ymin, xmax, ymax], 'category_id': int(c_id), 'area': (xmax-xmin)*(ymax-ymin)})
    return annotations

def calculate_ap(preds, gts, iou_threshold=0.5):
    if not gts: return 0.0
    if not preds: return 0.0
    preds = sorted(preds, key=lambda x: x['score'], reverse=True)
    tp, fp = np.zeros(len(preds)), np.zeros(len(preds))
    matched_gt = [False] * len(gts)
    for i, p in enumerate(preds):
        best_iou, best_idx = -1, -1
        for j, g in enumerate(gts):
            if not matched_gt[j] and p['category_id'] == g['category_id']:
                iou = bbox_iou(p['bbox'], g['bbox'])
                if iou > best_iou: best_iou, best_idx = iou, j
        if best_iou >= iou_threshold:
            tp[i] = 1
            matched_gt[best_idx] = True
        else: fp[i] = 1
    tp_cumsum, fp_cumsum = np.cumsum(tp), np.cumsum(fp)
    recalls = tp_cumsum / len(gts)
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)
    # 11-point interpolation
    ap = 0.0
    for t in np.arange(0, 1.1, 0.1):
        p = np.max(precisions[recalls >= t]) if any(recalls >= t) else 0.0
        ap += p / 11.0
    return ap

# --- MAIN EVALUATION LOOP ---
data_yaml = load_yaml(f"{BASE_PATH}/fold_0/data.yaml")
num_classes = len(data_yaml.get('names', []))
results_log = []

for folder in FOLDERS:
    print(f"\n--- 🚀 Starting Evaluation with mAP: {folder} ---")
    img_dir, label_dir = f"{BASE_PATH}/{folder}/valid/images", f"{BASE_PATH}/{folder}/valid/labels"
    image_filenames = [f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    detection_model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics', model_path=MODEL_MAP[folder],
        confidence_threshold=0.30, device='cuda:0'
    )

    f_tp, f_fp, f_fn = 0, 0, 0
    f_tp_s, f_fn_s = 0, 0
    all_preds_cls = {i: [] for i in range(num_classes)}
    all_gts_cls = {i: [] for i in range(num_classes)}

    for idx, img_name in enumerate(image_filenames):
        image_path = os.path.join(img_dir, img_name)
        dims = get_image_dimensions(image_path)
        gts = load_yolo_annotations(label_dir, img_name, dims)

        result = get_sliced_prediction(
            image_path, detection_model,
            slice_height=1024, slice_width=1024,
            overlap_height_ratio=0.2, overlap_width_ratio=0.2,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.3
        )

        preds = [{'bbox': obj.bbox.to_xyxy(), 'category_id': obj.category.id, 'score': obj.score.value}
                 for obj in result.object_prediction_list]

        # Simple counts for Precision/Recall
        matched_gt = [False] * len(gts)
        sorted_p = sorted(preds, key=lambda x: x['score'], reverse=True)
        for p in sorted_p:
            best_iou, best_idx = -1, -1
            for i, g in enumerate(gts):
                if not matched_gt[i] and p['category_id'] == g['category_id']:
                    iou = bbox_iou(p['bbox'], g['bbox'])
                    if iou > best_iou: best_iou, best_idx = iou, i
            if best_iou >= 0.5:
                f_tp += 1
                matched_gt[best_idx] = True
            else: f_fp += 1
        f_fn += matched_gt.count(False)

        # Store for mAP and Small Object metrics
        for p in preds: all_preds_cls[p['category_id']].append(p)
        for g in gts:
            all_gts_cls[g['category_id']].append(g)
            if g['area'] < SMALL_OBJ_THRESHOLD:
                # Local check for small recall
                best_iou = max([bbox_iou(p['bbox'], g['bbox']) for p in preds if p['category_id'] == g['category_id']] + [0])
                if best_iou >= 0.5: f_tp_s += 1
                else: f_fn_s += 1

        if idx < 3: # Save samples
            visual_result = visualize_object_predictions(np.array(read_image(image_path)), result.object_prediction_list)
            cv2.imwrite(os.path.join(OUTPUT_VIS_DIR, f"{folder}_{img_name}"), cv2.cvtColor(visual_result['image'], cv2.COLOR_RGB2BGR))

    # Fold Metrics
    p_f = f_tp / (f_tp + f_fp) if (f_tp + f_fp) > 0 else 0
    r_f = f_tp / (f_tp + f_fn) if (f_tp + f_fn) > 0 else 0
    f1_f = 2*(p_f*r_f)/(p_f+r_f) if (p_f+r_f)>0 else 0
    map50 = np.mean([calculate_ap(all_preds_cls[i], all_gts_cls[i]) for i in range(num_classes) if all_gts_cls[i]])
    s_rec = f_tp_s / (f_tp_s + f_fn_s) if (f_tp_s + f_fn_s) > 0 else 0

    results_log.append({
        'Folder': folder, 'mAP@0.5': round(map50, 4), 'Precision': round(p_f, 4),
        'Recall': round(r_f, 4), 'F1': round(f1_f, 4), 'Small_Recall': round(s_rec, 4)
    })

df = pd.DataFrame(results_log)

# Calculate average and standard deviation
avg_row = df.mean(numeric_only=True).to_dict()
avg_row['Folder'] = 'AVERAGE'

std_row = df.std(numeric_only=True).to_dict()
std_row['Folder'] = 'STD DEV'

df = pd.concat([df, pd.DataFrame([avg_row, std_row])], ignore_index=True)

print("\n--- Final Results (SAHI + mAP) ---")
print(df.to_string(index=False))



--- 🚀 Starting Evaluation with mAP: fold_0 ---
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Performing prediction on 6 slices.
Performing prediction on 9 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 9 slices.
Performing prediction on 40 slices.
Performing prediction on 6 slices.
Performing prediction on 3 slices.
Performing prediction on 6 slices.
Performing prediction on 3 slices.
Performing prediction on 1 slices.
Performing prediction on 4 slices.
Performing prediction on 20 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 20 slices.
Performi

###Subset: Missing Only

In [ ]:
import os
import numpy as np
import pandas as pd
import yaml
import time
import cv2
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import visualize_object_predictions, read_image

# --- CONFIGURATION ---
BASE_PATH = '/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data'
OUTPUT_VIS_DIR = '/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV11/missing_only/visuals'
FOLDERS = ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']
SMALL_OBJ_THRESHOLD = 32 * 32
os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)

MODEL_MAP = {
    'fold_0': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/missing_only/fold_0/weights/best.pt',
    'fold_1': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/missing_only/fold_1/weights/best.pt',
    'fold_2': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/missing_only/fold_2/weights/best.pt',
    'fold_3': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/missing_only/fold_3/weights/best.pt',
    'fold_4': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/missing_only/fold_4/weights/best.pt',
}

# --- HELPER FUNCTIONS ---
def get_image_dimensions(image_path):
    with Image.open(image_path) as img: return img.size

def load_yaml(yaml_path):
    with open(yaml_path, 'r') as f: return yaml.safe_load(f)

def bbox_iou(box1, box2):
    x1, y1, x2, y2 = max(box1[0], box2[0]), max(box1[1], box2[1]), min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def load_yolo_annotations(label_dir, image_filename, image_dims):
    annotations = []
    label_path = os.path.join(label_dir, os.path.splitext(image_filename)[0] + '.txt')
    if not os.path.exists(label_path): return annotations
    img_w, img_h = image_dims
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                c_id, cx, cy, w, h = map(float, parts)
                xmin, ymin = int((cx - w/2) * img_w), int((cy - h/2) * img_h)
                xmax, ymax = int((cx + w/2) * img_w), int((cy + h/2) * img_h)
                annotations.append({'bbox': [xmin, ymin, xmax, ymax], 'category_id': int(c_id), 'area': (xmax-xmin)*(ymax-ymin)})
    return annotations

def calculate_ap(preds, gts, iou_threshold=0.5):
    if not gts: return 0.0
    if not preds: return 0.0
    preds = sorted(preds, key=lambda x: x['score'], reverse=True)
    tp, fp = np.zeros(len(preds)), np.zeros(len(preds))
    matched_gt = [False] * len(gts)
    for i, p in enumerate(preds):
        best_iou, best_idx = -1, -1
        for j, g in enumerate(gts):
            if not matched_gt[j] and p['category_id'] == g['category_id']:
                iou = bbox_iou(p['bbox'], g['bbox'])
                if iou > best_iou: best_iou, best_idx = iou, j
        if best_iou >= iou_threshold:
            tp[i] = 1
            matched_gt[best_idx] = True
        else: fp[i] = 1
    tp_cumsum, fp_cumsum = np.cumsum(tp), np.cumsum(fp)
    recalls = tp_cumsum / len(gts)
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)
    # 11-point interpolation
    ap = 0.0
    for t in np.arange(0, 1.1, 0.1):
        p = np.max(precisions[recalls >= t]) if any(recalls >= t) else 0.0
        ap += p / 11.0
    return ap

# --- MAIN EVALUATION LOOP ---
data_yaml = load_yaml(f"{BASE_PATH}/fold_0/data.yaml")
num_classes = len(data_yaml.get('names', []))
results_log = []

for folder in FOLDERS:
    print(f"\n--- 🚀 Starting Evaluation with mAP: {folder} ---")
    img_dir, label_dir = f"{BASE_PATH}/{folder}/valid/images", f"{BASE_PATH}/{folder}/valid/labels"
    image_filenames = [f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    detection_model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics', model_path=MODEL_MAP[folder],
        confidence_threshold=0.30, device='cuda:0'
    )

    f_tp, f_fp, f_fn = 0, 0, 0
    f_tp_s, f_fn_s = 0, 0
    all_preds_cls = {i: [] for i in range(num_classes)}
    all_gts_cls = {i: [] for i in range(num_classes)}

    for idx, img_name in enumerate(image_filenames):
        image_path = os.path.join(img_dir, img_name)
        dims = get_image_dimensions(image_path)
        gts = load_yolo_annotations(label_dir, img_name, dims)

        result = get_sliced_prediction(
            image_path, detection_model,
            slice_height=1024, slice_width=1024,
            overlap_height_ratio=0.2, overlap_width_ratio=0.2,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.3
        )

        preds = [{'bbox': obj.bbox.to_xyxy(), 'category_id': obj.category.id, 'score': obj.score.value}
                 for obj in result.object_prediction_list]

        # Simple counts for Precision/Recall
        matched_gt = [False] * len(gts)
        sorted_p = sorted(preds, key=lambda x: x['score'], reverse=True)
        for p in sorted_p:
            best_iou, best_idx = -1, -1
            for i, g in enumerate(gts):
                if not matched_gt[i] and p['category_id'] == g['category_id']:
                    iou = bbox_iou(p['bbox'], g['bbox'])
                    if iou > best_iou: best_iou, best_idx = iou, i
            if best_iou >= 0.5:
                f_tp += 1
                matched_gt[best_idx] = True
            else: f_fp += 1
        f_fn += matched_gt.count(False)

        # Store for mAP and Small Object metrics
        for p in preds: all_preds_cls[p['category_id']].append(p)
        for g in gts:
            all_gts_cls[g['category_id']].append(g)
            if g['area'] < SMALL_OBJ_THRESHOLD:
                # Local check for small recall
                best_iou = max([bbox_iou(p['bbox'], g['bbox']) for p in preds if p['category_id'] == g['category_id']] + [0])
                if best_iou >= 0.5: f_tp_s += 1
                else: f_fn_s += 1

        if idx < 3: # Save samples
            visual_result = visualize_object_predictions(np.array(read_image(image_path)), result.object_prediction_list)
            cv2.imwrite(os.path.join(OUTPUT_VIS_DIR, f"{folder}_{img_name}"), cv2.cvtColor(visual_result['image'], cv2.COLOR_RGB2BGR))

    # Fold Metrics
    p_f = f_tp / (f_tp + f_fp) if (f_tp + f_fp) > 0 else 0
    r_f = f_tp / (f_tp + f_fn) if (f_tp + f_fn) > 0 else 0
    f1_f = 2*(p_f*r_f)/(p_f+r_f) if (p_f+r_f)>0 else 0
    map50 = np.mean([calculate_ap(all_preds_cls[i], all_gts_cls[i]) for i in range(num_classes) if all_gts_cls[i]])
    s_rec = f_tp_s / (f_tp_s + f_fn_s) if (f_tp_s + f_fn_s) > 0 else 0

    results_log.append({
        'Folder': folder, 'mAP@0.5': round(map50, 4), 'Precision': round(p_f, 4),
        'Recall': round(r_f, 4), 'F1': round(f1_f, 4), 'Small_Recall': round(s_rec, 4)
    })

df = pd.DataFrame(results_log)

# Calculate average and standard deviation
avg_row = df.mean(numeric_only=True).to_dict()
avg_row['Folder'] = 'AVERAGE'

std_row = df.std(numeric_only=True).to_dict()
std_row['Folder'] = 'STD DEV'

df = pd.concat([df, pd.DataFrame([avg_row, std_row])], ignore_index=True)

print("\n--- Final Results (SAHI + mAP) ---")
print(df.to_string(index=False))


--- 🚀 Starting Evaluation with mAP: fold_0 ---
Performing prediction on 6 slices.
Performing prediction on 35 slices.
Performing prediction on 4 slices.
Performing prediction on 12 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 1 slices.
Performing prediction on 4 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 4 slices.
Performing prediction on 1 slices.
Performing prediction on 4 slices.
Performing prediction on 6 slices.
Performing prediction on 1 slices.
Performing prediction on 6 slices.
Performing prediction on 8 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 6 slices.
Performing prediction on 2 slices.
Performing prediction on 2 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 2 slices.
Performing prediction on 3 slices.
Perfo

###Subset: Non Missing

In [ ]:
import os
import numpy as np
import pandas as pd
import yaml
import time
import cv2
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import visualize_object_predictions, read_image

# --- CONFIGURATION ---
BASE_PATH = '/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data'
OUTPUT_VIS_DIR = '/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV11/non_missing/visuals'
FOLDERS = ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']
SMALL_OBJ_THRESHOLD = 32 * 32
os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)

MODEL_MAP = {
    'fold_0': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/non_missing/fold_0/weights/best.pt',
    'fold_1': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/non_missing/fold_1/weights/best.pt',
    'fold_2': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/non_missing/fold_2/weights/best.pt',
    'fold_3': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/non_missing/fold_3/weights/best.pt',
    'fold_4': '/content/drive/MyDrive/PCB_MC/Results/YOLOV11/non_missing/fold_4/weights/best.pt',
}

# --- HELPER FUNCTIONS ---
def get_image_dimensions(image_path):
    with Image.open(image_path) as img: return img.size

def load_yaml(yaml_path):
    with open(yaml_path, 'r') as f: return yaml.safe_load(f)

def bbox_iou(box1, box2):
    x1, y1, x2, y2 = max(box1[0], box2[0]), max(box1[1], box2[1]), min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def load_yolo_annotations(label_dir, image_filename, image_dims):
    annotations = []
    label_path = os.path.join(label_dir, os.path.splitext(image_filename)[0] + '.txt')
    if not os.path.exists(label_path): return annotations
    img_w, img_h = image_dims
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                c_id, cx, cy, w, h = map(float, parts)
                xmin, ymin = int((cx - w/2) * img_w), int((cy - h/2) * img_h)
                xmax, ymax = int((cx + w/2) * img_w), int((cy + h/2) * img_h)
                annotations.append({'bbox': [xmin, ymin, xmax, ymax], 'category_id': int(c_id), 'area': (xmax-xmin)*(ymax-ymin)})
    return annotations

def calculate_ap(preds, gts, iou_threshold=0.5):
    if not gts: return 0.0
    if not preds: return 0.0
    preds = sorted(preds, key=lambda x: x['score'], reverse=True)
    tp, fp = np.zeros(len(preds)), np.zeros(len(preds))
    matched_gt = [False] * len(gts)
    for i, p in enumerate(preds):
        best_iou, best_idx = -1, -1
        for j, g in enumerate(gts):
            if not matched_gt[j] and p['category_id'] == g['category_id']:
                iou = bbox_iou(p['bbox'], g['bbox'])
                if iou > best_iou: best_iou, best_idx = iou, j
        if best_iou >= iou_threshold:
            tp[i] = 1
            matched_gt[best_idx] = True
        else: fp[i] = 1
    tp_cumsum, fp_cumsum = np.cumsum(tp), np.cumsum(fp)
    recalls = tp_cumsum / len(gts)
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)
    # 11-point interpolation
    ap = 0.0
    for t in np.arange(0, 1.1, 0.1):
        p = np.max(precisions[recalls >= t]) if any(recalls >= t) else 0.0
        ap += p / 11.0
    return ap

# --- MAIN EVALUATION LOOP ---
data_yaml = load_yaml(f"{BASE_PATH}/fold_0/data.yaml")
num_classes = len(data_yaml.get('names', []))
results_log = []

for folder in FOLDERS:
    print(f"\n--- 🚀 Starting Evaluation with mAP: {folder} ---")
    img_dir, label_dir = f"{BASE_PATH}/{folder}/valid/images", f"{BASE_PATH}/{folder}/valid/labels"
    image_filenames = [f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    detection_model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics', model_path=MODEL_MAP[folder],
        confidence_threshold=0.30, device='cuda:0'
    )

    f_tp, f_fp, f_fn = 0, 0, 0
    f_tp_s, f_fn_s = 0, 0
    all_preds_cls = {i: [] for i in range(num_classes)}
    all_gts_cls = {i: [] for i in range(num_classes)}

    for idx, img_name in enumerate(image_filenames):
        image_path = os.path.join(img_dir, img_name)
        dims = get_image_dimensions(image_path)
        gts = load_yolo_annotations(label_dir, img_name, dims)

        result = get_sliced_prediction(
            image_path, detection_model,
            slice_height=1024, slice_width=1024,
            overlap_height_ratio=0.2, overlap_width_ratio=0.2,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.3
        )

        preds = [{'bbox': obj.bbox.to_xyxy(), 'category_id': obj.category.id, 'score': obj.score.value}
                 for obj in result.object_prediction_list]

        # Simple counts for Precision/Recall
        matched_gt = [False] * len(gts)
        sorted_p = sorted(preds, key=lambda x: x['score'], reverse=True)
        for p in sorted_p:
            best_iou, best_idx = -1, -1
            for i, g in enumerate(gts):
                if not matched_gt[i] and p['category_id'] == g['category_id']:
                    iou = bbox_iou(p['bbox'], g['bbox'])
                    if iou > best_iou: best_iou, best_idx = iou, i
            if best_iou >= 0.5:
                f_tp += 1
                matched_gt[best_idx] = True
            else: f_fp += 1
        f_fn += matched_gt.count(False)

        # Store for mAP and Small Object metrics
        for p in preds: all_preds_cls[p['category_id']].append(p)
        for g in gts:
            all_gts_cls[g['category_id']].append(g)
            if g['area'] < SMALL_OBJ_THRESHOLD:
                # Local check for small recall
                best_iou = max([bbox_iou(p['bbox'], g['bbox']) for p in preds if p['category_id'] == g['category_id']] + [0])
                if best_iou >= 0.5: f_tp_s += 1
                else: f_fn_s += 1

        if idx < 3: # Save samples
            visual_result = visualize_object_predictions(np.array(read_image(image_path)), result.object_prediction_list)
            cv2.imwrite(os.path.join(OUTPUT_VIS_DIR, f"{folder}_{img_name}"), cv2.cvtColor(visual_result['image'], cv2.COLOR_RGB2BGR))

    # Fold Metrics
    p_f = f_tp / (f_tp + f_fp) if (f_tp + f_fp) > 0 else 0
    r_f = f_tp / (f_tp + f_fn) if (f_tp + f_fn) > 0 else 0
    f1_f = 2*(p_f*r_f)/(p_f+r_f) if (p_f+r_f)>0 else 0
    map50 = np.mean([calculate_ap(all_preds_cls[i], all_gts_cls[i]) for i in range(num_classes) if all_gts_cls[i]])
    s_rec = f_tp_s / (f_tp_s + f_fn_s) if (f_tp_s + f_fn_s) > 0 else 0

    results_log.append({
        'Folder': folder, 'mAP@0.5': round(map50, 4), 'Precision': round(p_f, 4),
        'Recall': round(r_f, 4), 'F1': round(f1_f, 4), 'Small_Recall': round(s_rec, 4)
    })

df = pd.DataFrame(results_log)

# Calculate average and standard deviation
avg_row = df.mean(numeric_only=True).to_dict()
avg_row['Folder'] = 'AVERAGE'

std_row = df.std(numeric_only=True).to_dict()
std_row['Folder'] = 'STD DEV'

df = pd.concat([df, pd.DataFrame([avg_row, std_row])], ignore_index=True)

print("\n--- Final Results (SAHI + mAP) ---")
print(df.to_string(index=False))


--- 🚀 Starting Evaluation with mAP: fold_0 ---
Performing prediction on 3 slices.
Performing prediction on 3 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 4 slices.
Performing prediction on 1 slices.
Performing prediction on 6 slices.
Performing prediction on 20 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 8 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 3 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 4 slices.
Performing prediction on 20 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 9 slices.
Perfo

## Visualizations

####Subset: Full Dataset + Visualization

In [ ]:
import os
import cv2
import yaml
import numpy as np
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# ============================================================
# CONFIG
# ============================================================

BASE_PATH = "/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data"
DATA_YAML = "/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_0/data.yaml"

OUT_ROOT = "/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV11/full_dataset/tp_fp_fn_split"

FOLDS = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]

MODEL_MAP = {
    "fold_0": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/full_dataset/fold_0/weights/best.pt",
    "fold_1": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/full_dataset/fold_1/weights/best.pt",
    "fold_2": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/full_dataset/fold_2/weights/best.pt",
    "fold_3": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/full_dataset/fold_3/weights/best.pt",
    "fold_4": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/full_dataset/fold_4/weights/best.pt",
}

os.makedirs(OUT_ROOT, exist_ok=True)

# SAHI
SLICE = 640
OVERLAP = 0.2

CONF_MODEL = 0.30
CONF_LOW = 0.80
IOU_TH = 0.5

SAVE_N = 50  # images per fold

# ============================================================
# COLORS (BGR)
# ============================================================

MAGENTA = (255, 0, 255)
YELLOW  = (0, 255, 255)
RED     = (0, 0, 255)
ORANGE  = (0, 165, 255)
WHITE   = (255, 255, 255)
BLACK   = (0, 0, 0)

# ============================================================
# LOAD CLASS INFO
# ============================================================

with open(DATA_YAML, "r") as f:
    data_yaml = yaml.safe_load(f)

ID2NAME = {i: n for i, n in enumerate(data_yaml["names"])}

CONNECTOR_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "connector" in n.lower()
}

MISSING_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "missing" in n.lower()
}

print("Connector IDs:", CONNECTOR_CLASS_IDS)
print("Missing IDs:", MISSING_CLASS_IDS)

# ============================================================
# HELPERS
# ============================================================

def get_image_dimensions(path):
    with Image.open(path) as im:
        return im.size


def draw_text(img, text, x, y):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, BLACK, 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, WHITE, 1, cv2.LINE_AA)


def bbox_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter == 0:
        return 0.0

    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])

    return inter / (areaA + areaB - inter + 1e-9)


def load_gt(label_dir, img_name, dims):
    gts = []

    path = os.path.join(
        label_dir,
        os.path.splitext(img_name)[0] + ".txt"
    )

    if not os.path.exists(path):
        return gts

    W, H = dims

    with open(path) as f:
        for line in f:
            c, cx, cy, w, h = map(float, line.split())

            x1 = (cx - w/2) * W
            y1 = (cy - h/2) * H
            x2 = (cx + w/2) * W
            y2 = (cy + h/2) * H

            gts.append({
                "category_id": int(c),
                "bbox": [x1, y1, x2, y2]
            })

    return gts


def match(preds, gts):

    preds = sorted(preds, key=lambda x: x["score"], reverse=True)
    used = [False]*len(gts)

    tp, fp = [], []

    for p in preds:

        best_iou = 0
        best_j = -1

        for j, g in enumerate(gts):

            if used[j]:
                continue

            if p["category_id"] != g["category_id"]:
                continue

            iou = bbox_iou(p["bbox"], g["bbox"])

            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= IOU_TH:
            used[best_j] = True
            tp.append(p)
        else:
            fp.append(p)

    fn = [gts[i] for i in range(len(gts)) if not used[i]]

    return tp, fp, fn


def class_color(cid, mode):

    if mode == "fp":
        return ORANGE

    if mode == "fn":
        return RED

    if cid in CONNECTOR_CLASS_IDS:
        return YELLOW

    return MAGENTA


def should_label(cid, score, mode):

    if cid in MISSING_CLASS_IDS:
        return True

    if mode in ["fp", "fn"]:
        return True

    if score is not None and score < CONF_LOW:
        return True

    return False


def draw_boxes(img, items, mode):

    for it in items:

        cid = it["category_id"]
        score = it.get("score")

        x1, y1, x2, y2 = map(int, it["bbox"])

        color = class_color(cid, mode)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        if should_label(cid, score, mode):

            name = ID2NAME[cid]

            if mode == "fp":
                text = f"FP {name}:{score:.2f}"
            elif mode == "fn":
                text = f"FN {name}"
            else:
                text = f"{name}:{score:.2f}"

            draw_text(img, text, x1, max(15, y1-5))


def save_views(img_path, tp, fp, fn, out_base):

    img = cv2.imread(img_path)

    tp_img = img.copy()
    fp_img = img.copy()
    fn_img = img.copy()

    draw_boxes(tp_img, tp, "tp")
    draw_boxes(fp_img, fp, "fp")
    draw_boxes(fn_img, fn, "fn")

    cv2.imwrite(out_base + "_TP.png", tp_img)
    cv2.imwrite(out_base + "_FP.png", fp_img)
    cv2.imwrite(out_base + "_FN.png", fn_img)


# ============================================================
# MAIN LOOP
# ============================================================

for fold in FOLDS:

    print(f"\n🚀 Processing {fold}")

    img_dir = f"{BASE_PATH}/{fold}/valid/images"
    label_dir = f"{BASE_PATH}/{fold}/valid/labels"

    out_dir = os.path.join(OUT_ROOT, fold)
    os.makedirs(out_dir, exist_ok=True)

    model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=MODEL_MAP[fold],
        confidence_threshold=CONF_MODEL,
        device="cuda:0"
    )

    images = sorted(os.listdir(img_dir))[:SAVE_N]

    for img_name in images:

        img_path = os.path.join(img_dir, img_name)

        dims = get_image_dimensions(img_path)
        gts = load_gt(label_dir, img_name, dims)

        result = get_sliced_prediction(
            img_path,
            model,
            slice_height=SLICE,
            slice_width=SLICE,
            overlap_height_ratio=OVERLAP,
            overlap_width_ratio=OVERLAP,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.5
        )

        preds = []

        for obj in result.object_prediction_list:

            x1, y1, x2, y2 = obj.bbox.to_xyxy()

            preds.append({
                "category_id": int(obj.category.id),
                "score": float(obj.score.value),
                "bbox": [x1, y1, x2, y2]
            })

        tp, fp, fn = match(preds, gts)

        out_base = os.path.join(
            out_dir,
            os.path.splitext(img_name)[0]
        )

        save_views(img_path, tp, fp, fn, out_base)

    print(f"✅ Saved to {out_dir}")

print("\nDONE ✅")


Connector IDs: {3}
Missing IDs: {23, 24, 25, 26, 27, 28, 29, 30}

🚀 Processing fold_0
Performing prediction on 2 slices.
Performing prediction on 2 slices.
Performing prediction on 12 slices.
Performing prediction on 40 slices.
Performing prediction on 30 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 12 slices.
Performing prediction on 2 slices.
Performing prediction on 12 slices.
Performing prediction on 8 slices.
Performing prediction on 12 slices.
Performing prediction on 8 slices.
Performing prediction on 8 slices.
Performing prediction on 40 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 1 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction o

####Subset: Components Only  + Visualization

In [ ]:
import os
import cv2
import yaml
import numpy as np
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# ============================================================
# CONFIG
# ============================================================

BASE_PATH = "/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data"
DATA_YAML = "/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_0/data.yaml"

OUT_ROOT = "/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV11/components_only/tp_fp_fn_split"

FOLDS = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]

MODEL_MAP = {
    "fold_0": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/components_only/fold_0/weights/best.pt",
    "fold_1": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/components_only/fold_1/weights/best.pt",
    "fold_2": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/components_only/fold_2/weights/best.pt",
    "fold_3": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/components_only/fold_3/weights/best.pt",
    "fold_4": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/components_only/fold_4/weights/best.pt",
}

os.makedirs(OUT_ROOT, exist_ok=True)

# SAHI
SLICE = 640
OVERLAP = 0.2

CONF_MODEL = 0.30
CONF_LOW = 0.80
IOU_TH = 0.5

SAVE_N = 50  # images per fold

# ============================================================
# COLORS (BGR)
# ============================================================

MAGENTA = (255, 0, 255)
YELLOW  = (0, 255, 255)
RED     = (0, 0, 255)
ORANGE  = (0, 165, 255)
WHITE   = (255, 255, 255)
BLACK   = (0, 0, 0)

# ============================================================
# LOAD CLASS INFO
# ============================================================

with open(DATA_YAML, "r") as f:
    data_yaml = yaml.safe_load(f)

ID2NAME = {i: n for i, n in enumerate(data_yaml["names"])}

CONNECTOR_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "connector" in n.lower()
}

MISSING_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "missing" in n.lower()
}

print("Connector IDs:", CONNECTOR_CLASS_IDS)
print("Missing IDs:", MISSING_CLASS_IDS)

# ============================================================
# HELPERS
# ============================================================

def get_image_dimensions(path):
    with Image.open(path) as im:
        return im.size


def draw_text(img, text, x, y):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, BLACK, 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, WHITE, 1, cv2.LINE_AA)


def bbox_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter == 0:
        return 0.0

    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])

    return inter / (areaA + areaB - inter + 1e-9)


def load_gt(label_dir, img_name, dims):
    gts = []

    path = os.path.join(
        label_dir,
        os.path.splitext(img_name)[0] + ".txt"
    )

    if not os.path.exists(path):
        return gts

    W, H = dims

    with open(path) as f:
        for line in f:
            c, cx, cy, w, h = map(float, line.split())

            x1 = (cx - w/2) * W
            y1 = (cy - h/2) * H
            x2 = (cx + w/2) * W
            y2 = (cy + h/2) * H

            gts.append({
                "category_id": int(c),
                "bbox": [x1, y1, x2, y2]
            })

    return gts


def match(preds, gts):

    preds = sorted(preds, key=lambda x: x["score"], reverse=True)
    used = [False]*len(gts)

    tp, fp = [], []

    for p in preds:

        best_iou = 0
        best_j = -1

        for j, g in enumerate(gts):

            if used[j]:
                continue

            if p["category_id"] != g["category_id"]:
                continue

            iou = bbox_iou(p["bbox"], g["bbox"])

            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= IOU_TH:
            used[best_j] = True
            tp.append(p)
        else:
            fp.append(p)

    fn = [gts[i] for i in range(len(gts)) if not used[i]]

    return tp, fp, fn


def class_color(cid, mode):

    if mode == "fp":
        return ORANGE

    if mode == "fn":
        return RED

    if cid in CONNECTOR_CLASS_IDS:
        return YELLOW

    return MAGENTA


def should_label(cid, score, mode):

    if cid in MISSING_CLASS_IDS:
        return True

    if mode in ["fp", "fn"]:
        return True

    if score is not None and score < CONF_LOW:
        return True

    return False


def draw_boxes(img, items, mode):

    for it in items:

        cid = it["category_id"]
        score = it.get("score")

        x1, y1, x2, y2 = map(int, it["bbox"])

        color = class_color(cid, mode)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        if should_label(cid, score, mode):

            name = ID2NAME[cid]

            if mode == "fp":
                text = f"FP {name}:{score:.2f}"
            elif mode == "fn":
                text = f"FN {name}"
            else:
                text = f"{name}:{score:.2f}"

            draw_text(img, text, x1, max(15, y1-5))


def save_views(img_path, tp, fp, fn, out_base):

    img = cv2.imread(img_path)

    tp_img = img.copy()
    fp_img = img.copy()
    fn_img = img.copy()

    draw_boxes(tp_img, tp, "tp")
    draw_boxes(fp_img, fp, "fp")
    draw_boxes(fn_img, fn, "fn")

    cv2.imwrite(out_base + "_TP.png", tp_img)
    cv2.imwrite(out_base + "_FP.png", fp_img)
    cv2.imwrite(out_base + "_FN.png", fn_img)


# ============================================================
# MAIN LOOP
# ============================================================

for fold in FOLDS:

    print(f"\n🚀 Processing {fold}")

    img_dir = f"{BASE_PATH}/{fold}/valid/images"
    label_dir = f"{BASE_PATH}/{fold}/valid/labels"

    out_dir = os.path.join(OUT_ROOT, fold)
    os.makedirs(out_dir, exist_ok=True)

    model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=MODEL_MAP[fold],
        confidence_threshold=CONF_MODEL,
        device="cuda:0"
    )

    images = sorted(os.listdir(img_dir))[:SAVE_N]

    for img_name in images:

        img_path = os.path.join(img_dir, img_name)

        dims = get_image_dimensions(img_path)
        gts = load_gt(label_dir, img_name, dims)

        result = get_sliced_prediction(
            img_path,
            model,
            slice_height=SLICE,
            slice_width=SLICE,
            overlap_height_ratio=OVERLAP,
            overlap_width_ratio=OVERLAP,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.5
        )

        preds = []

        for obj in result.object_prediction_list:

            x1, y1, x2, y2 = obj.bbox.to_xyxy()

            preds.append({
                "category_id": int(obj.category.id),
                "score": float(obj.score.value),
                "bbox": [x1, y1, x2, y2]
            })

        tp, fp, fn = match(preds, gts)

        out_base = os.path.join(
            out_dir,
            os.path.splitext(img_name)[0]
        )

        save_views(img_path, tp, fp, fn, out_base)

    print(f"✅ Saved to {out_dir}")

print("\nDONE ✅")


Connector IDs: {3}
Missing IDs: set()

🚀 Processing fold_0
Performing prediction on 2 slices.
Performing prediction on 2 slices.
Performing prediction on 40 slices.
Performing prediction on 12 slices.
Performing prediction on 30 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 48 slices.
Performing prediction on 2 slices.
Performing prediction on 12 slices.
Performing prediction on 40 slices.
Performing prediction on 48 slices.
Performing prediction on 8 slices.
Performing prediction on 8 slices.
Performing prediction on 12 slices.
Performing prediction on 40 slices.
Performing prediction on 40 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 1 slices.
Performing prediction on 12 slices.
Performing prediction on 12 slices.
Performing prediction on 48 slices.
Performing pr

####Subset: Missing Only  + Visualization

In [ ]:
import os
import cv2
import yaml
import numpy as np
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# ============================================================
# CONFIG
# ============================================================

BASE_PATH = "/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data"
DATA_YAML = "/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/data.yaml"

OUT_ROOT = "/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV11/missing_only/tp_fp_fn_split"

FOLDS = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]

MODEL_MAP = {
    "fold_0": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/missing_only/fold_0/weights/best.pt",
    "fold_1": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/missing_only/fold_1/weights/best.pt",
    "fold_2": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/missing_only/fold_2/weights/best.pt",
    "fold_3": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/missing_only/fold_3/weights/best.pt",
    "fold_4": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/missing_only/fold_4/weights/best.pt",
}

os.makedirs(OUT_ROOT, exist_ok=True)

# SAHI
SLICE = 640
OVERLAP = 0.2

CONF_MODEL = 0.30
CONF_LOW = 0.80
IOU_TH = 0.5

SAVE_N = 50  # images per fold

# ============================================================
# COLORS (BGR)
# ============================================================

MAGENTA = (255, 0, 255)
YELLOW  = (0, 255, 255)
RED     = (0, 0, 255)
ORANGE  = (0, 165, 255)
WHITE   = (255, 255, 255)
BLACK   = (0, 0, 0)

# ============================================================
# LOAD CLASS INFO
# ============================================================

with open(DATA_YAML, "r") as f:
    data_yaml = yaml.safe_load(f)

ID2NAME = {i: n for i, n in enumerate(data_yaml["names"])}

CONNECTOR_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "connector" in n.lower()
}

MISSING_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "missing" in n.lower()
}

print("Connector IDs:", CONNECTOR_CLASS_IDS)
print("Missing IDs:", MISSING_CLASS_IDS)

# ============================================================
# HELPERS
# ============================================================

def get_image_dimensions(path):
    with Image.open(path) as im:
        return im.size


def draw_text(img, text, x, y):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, BLACK, 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, WHITE, 1, cv2.LINE_AA)


def bbox_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter == 0:
        return 0.0

    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])

    return inter / (areaA + areaB - inter + 1e-9)


def load_gt(label_dir, img_name, dims):
    gts = []

    path = os.path.join(
        label_dir,
        os.path.splitext(img_name)[0] + ".txt"
    )

    if not os.path.exists(path):
        return gts

    W, H = dims

    with open(path) as f:
        for line in f:
            c, cx, cy, w, h = map(float, line.split())

            x1 = (cx - w/2) * W
            y1 = (cy - h/2) * H
            x2 = (cx + w/2) * W
            y2 = (cy + h/2) * H

            gts.append({
                "category_id": int(c),
                "bbox": [x1, y1, x2, y2]
            })

    return gts


def match(preds, gts):

    preds = sorted(preds, key=lambda x: x["score"], reverse=True)
    used = [False]*len(gts)

    tp, fp = [], []

    for p in preds:

        best_iou = 0
        best_j = -1

        for j, g in enumerate(gts):

            if used[j]:
                continue

            if p["category_id"] != g["category_id"]:
                continue

            iou = bbox_iou(p["bbox"], g["bbox"])

            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= IOU_TH:
            used[best_j] = True
            tp.append(p)
        else:
            fp.append(p)

    fn = [gts[i] for i in range(len(gts)) if not used[i]]

    return tp, fp, fn


def class_color(cid, mode):

    if mode == "fp":
        return ORANGE

    if mode == "fn":
        return RED

    if cid in CONNECTOR_CLASS_IDS:
        return YELLOW

    return MAGENTA


def should_label(cid, score, mode):

    if cid in MISSING_CLASS_IDS:
        return True

    if mode in ["fp", "fn"]:
        return True

    if score is not None and score < CONF_LOW:
        return True

    return False


def draw_boxes(img, items, mode):

    for it in items:

        cid = it["category_id"]
        score = it.get("score")

        x1, y1, x2, y2 = map(int, it["bbox"])

        color = class_color(cid, mode)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        if should_label(cid, score, mode):

            name = ID2NAME[cid]

            if mode == "fp":
                text = f"FP {name}:{score:.2f}"
            elif mode == "fn":
                text = f"FN {name}"
            else:
                text = f"{name}:{score:.2f}"

            draw_text(img, text, x1, max(15, y1-5))


def save_views(img_path, tp, fp, fn, out_base):

    img = cv2.imread(img_path)

    tp_img = img.copy()
    fp_img = img.copy()
    fn_img = img.copy()

    draw_boxes(tp_img, tp, "tp")
    draw_boxes(fp_img, fp, "fp")
    draw_boxes(fn_img, fn, "fn")

    cv2.imwrite(out_base + "_TP.png", tp_img)
    cv2.imwrite(out_base + "_FP.png", fp_img)
    cv2.imwrite(out_base + "_FN.png", fn_img)


# ============================================================
# MAIN LOOP
# ============================================================

for fold in FOLDS:

    print(f"\n🚀 Processing {fold}")

    img_dir = f"{BASE_PATH}/{fold}/valid/images"
    label_dir = f"{BASE_PATH}/{fold}/valid/labels"

    out_dir = os.path.join(OUT_ROOT, fold)
    os.makedirs(out_dir, exist_ok=True)

    model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=MODEL_MAP[fold],
        confidence_threshold=CONF_MODEL,
        device="cuda:0"
    )

    images = sorted(os.listdir(img_dir))[:SAVE_N]

    for img_name in images:

        img_path = os.path.join(img_dir, img_name)

        dims = get_image_dimensions(img_path)
        gts = load_gt(label_dir, img_name, dims)

        result = get_sliced_prediction(
            img_path,
            model,
            slice_height=SLICE,
            slice_width=SLICE,
            overlap_height_ratio=OVERLAP,
            overlap_width_ratio=OVERLAP,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.5
        )

        preds = []

        for obj in result.object_prediction_list:

            x1, y1, x2, y2 = obj.bbox.to_xyxy()

            preds.append({
                "category_id": int(obj.category.id),
                "score": float(obj.score.value),
                "bbox": [x1, y1, x2, y2]
            })

        tp, fp, fn = match(preds, gts)

        out_base = os.path.join(
            out_dir,
            os.path.splitext(img_name)[0]
        )

        save_views(img_path, tp, fp, fn, out_base)

    print(f"✅ Saved to {out_dir}")

print("\nDONE ✅")


####Subset: Non Missing  + Visualization

In [ ]:
import os
import cv2
import yaml
import numpy as np
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# ============================================================
# CONFIG
# ============================================================

BASE_PATH = "/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data"
DATA_YAML = "/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_0/data.yaml"

OUT_ROOT = "/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV11/non_missing/tp_fp_fn_split"

FOLDS = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]

MODEL_MAP = {
    "fold_0": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/non_missing/fold_0/weights/best.pt",
    "fold_1": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/non_missing/fold_1/weights/best.pt",
    "fold_2": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/non_missing/fold_2/weights/best.pt",
    "fold_3": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/non_missing/fold_3/weights/best.pt",
    "fold_4": "/content/drive/MyDrive/PCB_MC/Results/YOLOV11/non_missing/fold_4/weights/best.pt",
}

os.makedirs(OUT_ROOT, exist_ok=True)

# SAHI
SLICE = 640
OVERLAP = 0.2

CONF_MODEL = 0.30
CONF_LOW = 0.80
IOU_TH = 0.5

SAVE_N = 50  # images per fold

# ============================================================
# COLORS (BGR)
# ============================================================

MAGENTA = (255, 0, 255)
YELLOW  = (0, 255, 255)
RED     = (0, 0, 255)
ORANGE  = (0, 165, 255)
WHITE   = (255, 255, 255)
BLACK   = (0, 0, 0)

# ============================================================
# LOAD CLASS INFO
# ============================================================

with open(DATA_YAML, "r") as f:
    data_yaml = yaml.safe_load(f)

ID2NAME = {i: n for i, n in enumerate(data_yaml["names"])}

CONNECTOR_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "connector" in n.lower()
}

MISSING_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "missing" in n.lower()
}

print("Connector IDs:", CONNECTOR_CLASS_IDS)
print("Missing IDs:", MISSING_CLASS_IDS)

# ============================================================
# HELPERS
# ============================================================

def get_image_dimensions(path):
    with Image.open(path) as im:
        return im.size


def draw_text(img, text, x, y):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, BLACK, 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, WHITE, 1, cv2.LINE_AA)


def bbox_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter == 0:
        return 0.0

    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])

    return inter / (areaA + areaB - inter + 1e-9)


def load_gt(label_dir, img_name, dims):
    gts = []

    path = os.path.join(
        label_dir,
        os.path.splitext(img_name)[0] + ".txt"
    )

    if not os.path.exists(path):
        return gts

    W, H = dims

    with open(path) as f:
        for line in f:
            c, cx, cy, w, h = map(float, line.split())

            x1 = (cx - w/2) * W
            y1 = (cy - h/2) * H
            x2 = (cx + w/2) * W
            y2 = (cy + h/2) * H

            gts.append({
                "category_id": int(c),
                "bbox": [x1, y1, x2, y2]
            })

    return gts


def match(preds, gts):

    preds = sorted(preds, key=lambda x: x["score"], reverse=True)
    used = [False]*len(gts)

    tp, fp = [], []

    for p in preds:

        best_iou = 0
        best_j = -1

        for j, g in enumerate(gts):

            if used[j]:
                continue

            if p["category_id"] != g["category_id"]:
                continue

            iou = bbox_iou(p["bbox"], g["bbox"])

            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= IOU_TH:
            used[best_j] = True
            tp.append(p)
        else:
            fp.append(p)

    fn = [gts[i] for i in range(len(gts)) if not used[i]]

    return tp, fp, fn


def class_color(cid, mode):

    if mode == "fp":
        return ORANGE

    if mode == "fn":
        return RED

    if cid in CONNECTOR_CLASS_IDS:
        return YELLOW

    return MAGENTA


def should_label(cid, score, mode):

    if cid in MISSING_CLASS_IDS:
        return True

    if mode in ["fp", "fn"]:
        return True

    if score is not None and score < CONF_LOW:
        return True

    return False


def draw_boxes(img, items, mode):

    for it in items:

        cid = it["category_id"]
        score = it.get("score")

        x1, y1, x2, y2 = map(int, it["bbox"])

        color = class_color(cid, mode)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        if should_label(cid, score, mode):

            name = ID2NAME[cid]

            if mode == "fp":
                text = f"FP {name}:{score:.2f}"
            elif mode == "fn":
                text = f"FN {name}"
            else:
                text = f"{name}:{score:.2f}"

            draw_text(img, text, x1, max(15, y1-5))


def save_views(img_path, tp, fp, fn, out_base):

    img = cv2.imread(img_path)

    tp_img = img.copy()
    fp_img = img.copy()
    fn_img = img.copy()

    draw_boxes(tp_img, tp, "tp")
    draw_boxes(fp_img, fp, "fp")
    draw_boxes(fn_img, fn, "fn")

    cv2.imwrite(out_base + "_TP.png", tp_img)
    cv2.imwrite(out_base + "_FP.png", fp_img)
    cv2.imwrite(out_base + "_FN.png", fn_img)


# ============================================================
# MAIN LOOP
# ============================================================

for fold in FOLDS:

    print(f"\n🚀 Processing {fold}")

    img_dir = f"{BASE_PATH}/{fold}/valid/images"
    label_dir = f"{BASE_PATH}/{fold}/valid/labels"

    out_dir = os.path.join(OUT_ROOT, fold)
    os.makedirs(out_dir, exist_ok=True)

    model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=MODEL_MAP[fold],
        confidence_threshold=CONF_MODEL,
        device="cuda:0"
    )

    images = sorted(os.listdir(img_dir))[:SAVE_N]

    for img_name in images:

        img_path = os.path.join(img_dir, img_name)

        dims = get_image_dimensions(img_path)
        gts = load_gt(label_dir, img_name, dims)

        result = get_sliced_prediction(
            img_path,
            model,
            slice_height=SLICE,
            slice_width=SLICE,
            overlap_height_ratio=OVERLAP,
            overlap_width_ratio=OVERLAP,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.5
        )

        preds = []

        for obj in result.object_prediction_list:

            x1, y1, x2, y2 = obj.bbox.to_xyxy()

            preds.append({
                "category_id": int(obj.category.id),
                "score": float(obj.score.value),
                "bbox": [x1, y1, x2, y2]
            })

        tp, fp, fn = match(preds, gts)

        out_base = os.path.join(
            out_dir,
            os.path.splitext(img_name)[0]
        )

        save_views(img_path, tp, fp, fn, out_base)

    print(f"✅ Saved to {out_dir}")

print("\nDONE ✅")


# YOLO V26 + SAHI

##Training

###Subset: Full Dataset

In [6]:
import os
import numpy as np
import pandas as pd
import yaml
import time
import cv2
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import visualize_object_predictions, read_image

# --- CONFIGURATION ---
BASE_PATH = '/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data'
OUTPUT_VIS_DIR = '/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV26/full_dataset/visuals'
FOLDERS = ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']
SMALL_OBJ_THRESHOLD = 32 * 32
os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)

MODEL_MAP = {
    'fold_0': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/full_dataset/fold_0/weights/best.pt',
    'fold_1': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/full_dataset/fold_1/weights/best.pt',
    'fold_2': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/full_dataset/fold_2/weights/best.pt',
    'fold_3': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/full_dataset/fold_3/weights/best.pt',
    'fold_4': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/full_dataset/fold_4/weights/best.pt',
}

# --- HELPER FUNCTIONS ---
def get_image_dimensions(image_path):
    with Image.open(image_path) as img: return img.size

def load_yaml(yaml_path):
    with open(yaml_path, 'r') as f: return yaml.safe_load(f)

def bbox_iou(box1, box2):
    x1, y1, x2, y2 = max(box1[0], box2[0]), max(box1[1], box2[1]), min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def load_yolo_annotations(label_dir, image_filename, image_dims):
    annotations = []
    label_path = os.path.join(label_dir, os.path.splitext(image_filename)[0] + '.txt')
    if not os.path.exists(label_path): return annotations
    img_w, img_h = image_dims
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                c_id, cx, cy, w, h = map(float, parts)
                xmin, ymin = int((cx - w/2) * img_w), int((cy - h/2) * img_h)
                xmax, ymax = int((cx + w/2) * img_w), int((cy + h/2) * img_h)
                annotations.append({'bbox': [xmin, ymin, xmax, ymax], 'category_id': int(c_id), 'area': (xmax-xmin)*(ymax-ymin)})
    return annotations

def calculate_ap(preds, gts, iou_threshold=0.5):
    if not gts: return 0.0
    if not preds: return 0.0
    preds = sorted(preds, key=lambda x: x['score'], reverse=True)
    tp, fp = np.zeros(len(preds)), np.zeros(len(preds))
    matched_gt = [False] * len(gts)
    for i, p in enumerate(preds):
        best_iou, best_idx = -1, -1
        for j, g in enumerate(gts):
            if not matched_gt[j] and p['category_id'] == g['category_id']:
                iou = bbox_iou(p['bbox'], g['bbox'])
                if iou > best_iou: best_iou, best_idx = iou, j
        if best_iou >= iou_threshold:
            tp[i] = 1
            matched_gt[best_idx] = True
        else: fp[i] = 1
    tp_cumsum, fp_cumsum = np.cumsum(tp), np.cumsum(fp)
    recalls = tp_cumsum / len(gts)
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)
    # 11-point interpolation
    ap = 0.0
    for t in np.arange(0, 1.1, 0.1):
        p = np.max(precisions[recalls >= t]) if any(recalls >= t) else 0.0
        ap += p / 11.0
    return ap

# --- MAIN EVALUATION LOOP ---
data_yaml = load_yaml(f"{BASE_PATH}/fold_0/data.yaml")
num_classes = len(data_yaml.get('names', []))
results_log = []

for folder in FOLDERS:
    print(f"\n--- 🚀 Starting Evaluation with mAP: {folder} ---")
    img_dir, label_dir = f"{BASE_PATH}/{folder}/valid/images", f"{BASE_PATH}/{folder}/valid/labels"
    image_filenames = [f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    detection_model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics', model_path=MODEL_MAP[folder],
        confidence_threshold=0.30, device='cuda:0'
    )

    f_tp, f_fp, f_fn = 0, 0, 0
    f_tp_s, f_fn_s = 0, 0
    all_preds_cls = {i: [] for i in range(num_classes)}
    all_gts_cls = {i: [] for i in range(num_classes)}

    for idx, img_name in enumerate(image_filenames):
        image_path = os.path.join(img_dir, img_name)
        dims = get_image_dimensions(image_path)
        gts = load_yolo_annotations(label_dir, img_name, dims)

        result = get_sliced_prediction(
            image_path, detection_model,
            slice_height=1024, slice_width=1024,
            overlap_height_ratio=0.2, overlap_width_ratio=0.2,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.3
        )

        preds = [{'bbox': obj.bbox.to_xyxy(), 'category_id': obj.category.id, 'score': obj.score.value}
                 for obj in result.object_prediction_list]

        # Simple counts for Precision/Recall
        matched_gt = [False] * len(gts)
        sorted_p = sorted(preds, key=lambda x: x['score'], reverse=True)
        for p in sorted_p:
            best_iou, best_idx = -1, -1
            for i, g in enumerate(gts):
                if not matched_gt[i] and p['category_id'] == g['category_id']:
                    iou = bbox_iou(p['bbox'], g['bbox'])
                    if iou > best_iou: best_iou, best_idx = iou, i
            if best_iou >= 0.5:
                f_tp += 1
                matched_gt[best_idx] = True
            else: f_fp += 1
        f_fn += matched_gt.count(False)

        # Store for mAP and Small Object metrics
        for p in preds: all_preds_cls[p['category_id']].append(p)
        for g in gts:
            all_gts_cls[g['category_id']].append(g)
            if g['area'] < SMALL_OBJ_THRESHOLD:
                # Local check for small recall
                best_iou = max([bbox_iou(p['bbox'], g['bbox']) for p in preds if p['category_id'] == g['category_id']] + [0])
                if best_iou >= 0.5: f_tp_s += 1
                else: f_fn_s += 1

        if idx < 3: # Save samples
            visual_result = visualize_object_predictions(np.array(read_image(image_path)), result.object_prediction_list)
            cv2.imwrite(os.path.join(OUTPUT_VIS_DIR, f"{folder}_{img_name}"), cv2.cvtColor(visual_result['image'], cv2.COLOR_RGB2BGR))

    # Fold Metrics
    p_f = f_tp / (f_tp + f_fp) if (f_tp + f_fp) > 0 else 0
    r_f = f_tp / (f_tp + f_fn) if (f_tp + f_fn) > 0 else 0
    f1_f = 2*(p_f*r_f)/(p_f+r_f) if (p_f+r_f)>0 else 0
    map50 = np.mean([calculate_ap(all_preds_cls[i], all_gts_cls[i]) for i in range(num_classes) if all_gts_cls[i]])
    s_rec = f_tp_s / (f_tp_s + f_fn_s) if (f_tp_s + f_fn_s) > 0 else 0

    results_log.append({
        'Folder': folder, 'mAP@0.5': round(map50, 4), 'Precision': round(p_f, 4),
        'Recall': round(r_f, 4), 'F1': round(f1_f, 4), 'Small_Recall': round(s_rec, 4)
    })

df = pd.DataFrame(results_log)

# Calculate average and standard deviation
avg_row = df.mean(numeric_only=True).to_dict()
avg_row['Folder'] = 'AVERAGE'

std_row = df.std(numeric_only=True).to_dict()
std_row['Folder'] = 'STD DEV'

df = pd.concat([df, pd.DataFrame([avg_row, std_row])], ignore_index=True)

print("\n--- Final Results (SAHI + mAP) ---")
print(df.to_string(index=False))


--- 🚀 Starting Evaluation with mAP: fold_0 ---
Performing prediction on 9 slices.
Performing prediction on 40 slices.
Performing prediction on 6 slices.
Performing prediction on 3 slices.
Performing prediction on 3 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 9 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 20 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 4 slices.
Performing prediction on 6 slices.
Performing prediction on 20 slices.
Performing prediction on 35 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 20 slices.
Performing prediction on 6 slices.
Pe

###Subset: Components Only

In [1]:
import os
import numpy as np
import pandas as pd
import yaml
import time
import cv2
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import visualize_object_predictions, read_image

# --- CONFIGURATION ---
BASE_PATH = '/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data'
OUTPUT_VIS_DIR = '/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV26/components_only/visuals'
FOLDERS = ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']
SMALL_OBJ_THRESHOLD = 32 * 32
os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)

MODEL_MAP = {
    'fold_0': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/components_only/fold_0/weights/best.pt',
    'fold_1': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/components_only/fold_1/weights/best.pt',
    'fold_2': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/components_only/fold_2/weights/best.pt',
    'fold_3': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/components_only/fold_3/weights/best.pt',
    'fold_4': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/components_only/fold_4/weights/best.pt',
}

# --- HELPER FUNCTIONS ---
def get_image_dimensions(image_path):
    with Image.open(image_path) as img: return img.size

def load_yaml(yaml_path):
    with open(yaml_path, 'r') as f: return yaml.safe_load(f)

def bbox_iou(box1, box2):
    x1, y1, x2, y2 = max(box1[0], box2[0]), max(box1[1], box2[1]), min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def load_yolo_annotations(label_dir, image_filename, image_dims):
    annotations = []
    label_path = os.path.join(label_dir, os.path.splitext(image_filename)[0] + '.txt')
    if not os.path.exists(label_path): return annotations
    img_w, img_h = image_dims
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                c_id, cx, cy, w, h = map(float, parts)
                xmin, ymin = int((cx - w/2) * img_w), int((cy - h/2) * img_h)
                xmax, ymax = int((cx + w/2) * img_w), int((cy + h/2) * img_h)
                annotations.append({'bbox': [xmin, ymin, xmax, ymax], 'category_id': int(c_id), 'area': (xmax-xmin)*(ymax-ymin)})
    return annotations

def calculate_ap(preds, gts, iou_threshold=0.5):
    if not gts: return 0.0
    if not preds: return 0.0
    preds = sorted(preds, key=lambda x: x['score'], reverse=True)
    tp, fp = np.zeros(len(preds)), np.zeros(len(preds))
    matched_gt = [False] * len(gts)
    for i, p in enumerate(preds):
        best_iou, best_idx = -1, -1
        for j, g in enumerate(gts):
            if not matched_gt[j] and p['category_id'] == g['category_id']:
                iou = bbox_iou(p['bbox'], g['bbox'])
                if iou > best_iou: best_iou, best_idx = iou, j
        if best_iou >= iou_threshold:
            tp[i] = 1
            matched_gt[best_idx] = True
        else: fp[i] = 1
    tp_cumsum, fp_cumsum = np.cumsum(tp), np.cumsum(fp)
    recalls = tp_cumsum / len(gts)
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)
    # 11-point interpolation
    ap = 0.0
    for t in np.arange(0, 1.1, 0.1):
        p = np.max(precisions[recalls >= t]) if any(recalls >= t) else 0.0
        ap += p / 11.0
    return ap

# --- MAIN EVALUATION LOOP ---
data_yaml = load_yaml(f"{BASE_PATH}/fold_0/data.yaml")
num_classes = len(data_yaml.get('names', []))
results_log = []

for folder in FOLDERS:
    print(f"\n--- 🚀 Starting Evaluation with mAP: {folder} ---")
    img_dir, label_dir = f"{BASE_PATH}/{folder}/valid/images", f"{BASE_PATH}/{folder}/valid/labels"
    image_filenames = [f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    detection_model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics', model_path=MODEL_MAP[folder],
        confidence_threshold=0.30, device='cuda:0'
    )

    f_tp, f_fp, f_fn = 0, 0, 0
    f_tp_s, f_fn_s = 0, 0
    all_preds_cls = {i: [] for i in range(num_classes)}
    all_gts_cls = {i: [] for i in range(num_classes)}

    for idx, img_name in enumerate(image_filenames):
        image_path = os.path.join(img_dir, img_name)
        dims = get_image_dimensions(image_path)
        gts = load_yolo_annotations(label_dir, img_name, dims)

        result = get_sliced_prediction(
            image_path, detection_model,
            slice_height=1024, slice_width=1024,
            overlap_height_ratio=0.2, overlap_width_ratio=0.2,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.3
        )

        preds = [{'bbox': obj.bbox.to_xyxy(), 'category_id': obj.category.id, 'score': obj.score.value}
                 for obj in result.object_prediction_list]

        # Simple counts for Precision/Recall
        matched_gt = [False] * len(gts)
        sorted_p = sorted(preds, key=lambda x: x['score'], reverse=True)
        for p in sorted_p:
            best_iou, best_idx = -1, -1
            for i, g in enumerate(gts):
                if not matched_gt[i] and p['category_id'] == g['category_id']:
                    iou = bbox_iou(p['bbox'], g['bbox'])
                    if iou > best_iou: best_iou, best_idx = iou, i
            if best_iou >= 0.5:
                f_tp += 1
                matched_gt[best_idx] = True
            else: f_fp += 1
        f_fn += matched_gt.count(False)

        # Store for mAP and Small Object metrics
        for p in preds: all_preds_cls[p['category_id']].append(p)
        for g in gts:
            all_gts_cls[g['category_id']].append(g)
            if g['area'] < SMALL_OBJ_THRESHOLD:
                # Local check for small recall
                best_iou = max([bbox_iou(p['bbox'], g['bbox']) for p in preds if p['category_id'] == g['category_id']] + [0])
                if best_iou >= 0.5: f_tp_s += 1
                else: f_fn_s += 1

        if idx < 3: # Save samples
            visual_result = visualize_object_predictions(np.array(read_image(image_path)), result.object_prediction_list)
            cv2.imwrite(os.path.join(OUTPUT_VIS_DIR, f"{folder}_{img_name}"), cv2.cvtColor(visual_result['image'], cv2.COLOR_RGB2BGR))

    # Fold Metrics
    p_f = f_tp / (f_tp + f_fp) if (f_tp + f_fp) > 0 else 0
    r_f = f_tp / (f_tp + f_fn) if (f_tp + f_fn) > 0 else 0
    f1_f = 2*(p_f*r_f)/(p_f+r_f) if (p_f+r_f)>0 else 0
    map50 = np.mean([calculate_ap(all_preds_cls[i], all_gts_cls[i]) for i in range(num_classes) if all_gts_cls[i]])
    s_rec = f_tp_s / (f_tp_s + f_fn_s) if (f_tp_s + f_fn_s) > 0 else 0

    results_log.append({
        'Folder': folder, 'mAP@0.5': round(map50, 4), 'Precision': round(p_f, 4),
        'Recall': round(r_f, 4), 'F1': round(f1_f, 4), 'Small_Recall': round(s_rec, 4)
    })

df = pd.DataFrame(results_log)

# Calculate average and standard deviation
avg_row = df.mean(numeric_only=True).to_dict()
avg_row['Folder'] = 'AVERAGE'

std_row = df.std(numeric_only=True).to_dict()
std_row['Folder'] = 'STD DEV'

df = pd.concat([df, pd.DataFrame([avg_row, std_row])], ignore_index=True)

print("\n--- Final Results (SAHI + mAP) ---")
print(df.to_string(index=False))

ModuleNotFoundError: No module named 'sahi'

###Subset: Missing Only

In [ ]:
import os
import numpy as np
import pandas as pd
import yaml
import time
import cv2
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import visualize_object_predictions, read_image

# --- CONFIGURATION ---
BASE_PATH = '/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data'
OUTPUT_VIS_DIR = '/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV26/missing_only/visuals'
FOLDERS = ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']
SMALL_OBJ_THRESHOLD = 32 * 32
os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)

MODEL_MAP = {
    'fold_0': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/missing_only/fold_0/weights/best.pt',
    'fold_1': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/missing_only/fold_1/weights/best.pt',
    'fold_2': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/missing_only/fold_2/weights/best.pt',
    'fold_3': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/missing_only/fold_3/weights/best.pt',
    'fold_4': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/missing_only/fold_4/weights/best.pt',
}

# --- HELPER FUNCTIONS ---
def get_image_dimensions(image_path):
    with Image.open(image_path) as img: return img.size

def load_yaml(yaml_path):
    with open(yaml_path, 'r') as f: return yaml.safe_load(f)

def bbox_iou(box1, box2):
    x1, y1, x2, y2 = max(box1[0], box2[0]), max(box1[1], box2[1]), min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def load_yolo_annotations(label_dir, image_filename, image_dims):
    annotations = []
    label_path = os.path.join(label_dir, os.path.splitext(image_filename)[0] + '.txt')
    if not os.path.exists(label_path): return annotations
    img_w, img_h = image_dims
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                c_id, cx, cy, w, h = map(float, parts)
                xmin, ymin = int((cx - w/2) * img_w), int((cy - h/2) * img_h)
                xmax, ymax = int((cx + w/2) * img_w), int((cy + h/2) * img_h)
                annotations.append({'bbox': [xmin, ymin, xmax, ymax], 'category_id': int(c_id), 'area': (xmax-xmin)*(ymax-ymin)})
    return annotations

def calculate_ap(preds, gts, iou_threshold=0.5):
    if not gts: return 0.0
    if not preds: return 0.0
    preds = sorted(preds, key=lambda x: x['score'], reverse=True)
    tp, fp = np.zeros(len(preds)), np.zeros(len(preds))
    matched_gt = [False] * len(gts)
    for i, p in enumerate(preds):
        best_iou, best_idx = -1, -1
        for j, g in enumerate(gts):
            if not matched_gt[j] and p['category_id'] == g['category_id']:
                iou = bbox_iou(p['bbox'], g['bbox'])
                if iou > best_iou: best_iou, best_idx = iou, j
        if best_iou >= iou_threshold:
            tp[i] = 1
            matched_gt[best_idx] = True
        else: fp[i] = 1
    tp_cumsum, fp_cumsum = np.cumsum(tp), np.cumsum(fp)
    recalls = tp_cumsum / len(gts)
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)
    # 11-point interpolation
    ap = 0.0
    for t in np.arange(0, 1.1, 0.1):
        p = np.max(precisions[recalls >= t]) if any(recalls >= t) else 0.0
        ap += p / 11.0
    return ap

# --- MAIN EVALUATION LOOP ---
data_yaml = load_yaml(f"{BASE_PATH}/fold_0/data.yaml")
num_classes = len(data_yaml.get('names', []))
results_log = []

for folder in FOLDERS:
    print(f"\n--- 🚀 Starting Evaluation with mAP: {folder} ---")
    img_dir, label_dir = f"{BASE_PATH}/{folder}/valid/images", f"{BASE_PATH}/{folder}/valid/labels"
    image_filenames = [f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    detection_model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics', model_path=MODEL_MAP[folder],
        confidence_threshold=0.30, device='cuda:0'
    )

    f_tp, f_fp, f_fn = 0, 0, 0
    f_tp_s, f_fn_s = 0, 0
    all_preds_cls = {i: [] for i in range(num_classes)}
    all_gts_cls = {i: [] for i in range(num_classes)}

    for idx, img_name in enumerate(image_filenames):
        image_path = os.path.join(img_dir, img_name)
        dims = get_image_dimensions(image_path)
        gts = load_yolo_annotations(label_dir, img_name, dims)

        result = get_sliced_prediction(
            image_path, detection_model,
            slice_height=1024, slice_width=1024,
            overlap_height_ratio=0.2, overlap_width_ratio=0.2,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.3
        )

        preds = [{'bbox': obj.bbox.to_xyxy(), 'category_id': obj.category.id, 'score': obj.score.value}
                 for obj in result.object_prediction_list]

        # Simple counts for Precision/Recall
        matched_gt = [False] * len(gts)
        sorted_p = sorted(preds, key=lambda x: x['score'], reverse=True)
        for p in sorted_p:
            best_iou, best_idx = -1, -1
            for i, g in enumerate(gts):
                if not matched_gt[i] and p['category_id'] == g['category_id']:
                    iou = bbox_iou(p['bbox'], g['bbox'])
                    if iou > best_iou: best_iou, best_idx = iou, i
            if best_iou >= 0.5:
                f_tp += 1
                matched_gt[best_idx] = True
            else: f_fp += 1
        f_fn += matched_gt.count(False)

        # Store for mAP and Small Object metrics
        for p in preds: all_preds_cls[p['category_id']].append(p)
        for g in gts:
            all_gts_cls[g['category_id']].append(g)
            if g['area'] < SMALL_OBJ_THRESHOLD:
                # Local check for small recall
                best_iou = max([bbox_iou(p['bbox'], g['bbox']) for p in preds if p['category_id'] == g['category_id']] + [0])
                if best_iou >= 0.5: f_tp_s += 1
                else: f_fn_s += 1

        if idx < 3: # Save samples
            visual_result = visualize_object_predictions(np.array(read_image(image_path)), result.object_prediction_list)
            cv2.imwrite(os.path.join(OUTPUT_VIS_DIR, f"{folder}_{img_name}"), cv2.cvtColor(visual_result['image'], cv2.COLOR_RGB2BGR))

    # Fold Metrics
    p_f = f_tp / (f_tp + f_fp) if (f_tp + f_fp) > 0 else 0
    r_f = f_tp / (f_tp + f_fn) if (f_tp + f_fn) > 0 else 0
    f1_f = 2*(p_f*r_f)/(p_f+r_f) if (p_f+r_f)>0 else 0
    map50 = np.mean([calculate_ap(all_preds_cls[i], all_gts_cls[i]) for i in range(num_classes) if all_gts_cls[i]])
    s_rec = f_tp_s / (f_tp_s + f_fn_s) if (f_tp_s + f_fn_s) > 0 else 0

    results_log.append({
        'Folder': folder, 'mAP@0.5': round(map50, 4), 'Precision': round(p_f, 4),
        'Recall': round(r_f, 4), 'F1': round(f1_f, 4), 'Small_Recall': round(s_rec, 4)
    })

df = pd.DataFrame(results_log)

# Calculate average and standard deviation
avg_row = df.mean(numeric_only=True).to_dict()
avg_row['Folder'] = 'AVERAGE'

std_row = df.std(numeric_only=True).to_dict()
std_row['Folder'] = 'STD DEV'

df = pd.concat([df, pd.DataFrame([avg_row, std_row])], ignore_index=True)

print("\n--- Final Results (SAHI + mAP) ---")
print(df.to_string(index=False))


--- 🚀 Starting Evaluation with mAP: fold_0 ---
Performing prediction on 6 slices.
Performing prediction on 35 slices.
Performing prediction on 4 slices.
Performing prediction on 12 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 1 slices.
Performing prediction on 4 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 4 slices.
Performing prediction on 1 slices.
Performing prediction on 4 slices.
Performing prediction on 6 slices.
Performing prediction on 1 slices.
Performing prediction on 6 slices.
Performing prediction on 8 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 6 slices.
Performing prediction on 2 slices.
Performing prediction on 2 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 2 slices.
Performing prediction on 3 slices.
Perfo

###Subset: Non Missing

In [ ]:
import os
import numpy as np
import pandas as pd
import yaml
import time
import cv2
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import visualize_object_predictions, read_image

# --- CONFIGURATION ---
BASE_PATH = '/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data'
OUTPUT_VIS_DIR = '/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV26/non_missing/visuals'
FOLDERS = ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']
SMALL_OBJ_THRESHOLD = 32 * 32
os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)

MODEL_MAP = {
    'fold_0': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/non_missing/fold_0/weights/best.pt',
    'fold_1': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/non_missing/fold_1/weights/best.pt',
    'fold_2': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/non_missing/fold_2/weights/best.pt',
    'fold_3': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/non_missing/fold_3/weights/best.pt',
    'fold_4': '/content/drive/MyDrive/PCB_MC/Results/YOLOV26/non_missing/fold_4/weights/best.pt',
}

# --- HELPER FUNCTIONS ---
def get_image_dimensions(image_path):
    with Image.open(image_path) as img: return img.size

def load_yaml(yaml_path):
    with open(yaml_path, 'r') as f: return yaml.safe_load(f)

def bbox_iou(box1, box2):
    x1, y1, x2, y2 = max(box1[0], box2[0]), max(box1[1], box2[1]), min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / union if union > 0 else 0

def load_yolo_annotations(label_dir, image_filename, image_dims):
    annotations = []
    label_path = os.path.join(label_dir, os.path.splitext(image_filename)[0] + '.txt')
    if not os.path.exists(label_path): return annotations
    img_w, img_h = image_dims
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                c_id, cx, cy, w, h = map(float, parts)
                xmin, ymin = int((cx - w/2) * img_w), int((cy - h/2) * img_h)
                xmax, ymax = int((cx + w/2) * img_w), int((cy + h/2) * img_h)
                annotations.append({'bbox': [xmin, ymin, xmax, ymax], 'category_id': int(c_id), 'area': (xmax-xmin)*(ymax-ymin)})
    return annotations

def calculate_ap(preds, gts, iou_threshold=0.5):
    if not gts: return 0.0
    if not preds: return 0.0
    preds = sorted(preds, key=lambda x: x['score'], reverse=True)
    tp, fp = np.zeros(len(preds)), np.zeros(len(preds))
    matched_gt = [False] * len(gts)
    for i, p in enumerate(preds):
        best_iou, best_idx = -1, -1
        for j, g in enumerate(gts):
            if not matched_gt[j] and p['category_id'] == g['category_id']:
                iou = bbox_iou(p['bbox'], g['bbox'])
                if iou > best_iou: best_iou, best_idx = iou, j
        if best_iou >= iou_threshold:
            tp[i] = 1
            matched_gt[best_idx] = True
        else: fp[i] = 1
    tp_cumsum, fp_cumsum = np.cumsum(tp), np.cumsum(fp)
    recalls = tp_cumsum / len(gts)
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)
    # 11-point interpolation
    ap = 0.0
    for t in np.arange(0, 1.1, 0.1):
        p = np.max(precisions[recalls >= t]) if any(recalls >= t) else 0.0
        ap += p / 11.0
    return ap

# --- MAIN EVALUATION LOOP ---
data_yaml = load_yaml(f"{BASE_PATH}/fold_0/data.yaml")
num_classes = len(data_yaml.get('names', []))
results_log = []

for folder in FOLDERS:
    print(f"\n--- 🚀 Starting Evaluation with mAP: {folder} ---")
    img_dir, label_dir = f"{BASE_PATH}/{folder}/valid/images", f"{BASE_PATH}/{folder}/valid/labels"
    image_filenames = [f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    detection_model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics', model_path=MODEL_MAP[folder],
        confidence_threshold=0.30, device='cuda:0'
    )

    f_tp, f_fp, f_fn = 0, 0, 0
    f_tp_s, f_fn_s = 0, 0
    all_preds_cls = {i: [] for i in range(num_classes)}
    all_gts_cls = {i: [] for i in range(num_classes)}

    for idx, img_name in enumerate(image_filenames):
        image_path = os.path.join(img_dir, img_name)
        dims = get_image_dimensions(image_path)
        gts = load_yolo_annotations(label_dir, img_name, dims)

        result = get_sliced_prediction(
            image_path, detection_model,
            slice_height=1024, slice_width=1024,
            overlap_height_ratio=0.2, overlap_width_ratio=0.2,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.3
        )

        preds = [{'bbox': obj.bbox.to_xyxy(), 'category_id': obj.category.id, 'score': obj.score.value}
                 for obj in result.object_prediction_list]

        # Simple counts for Precision/Recall
        matched_gt = [False] * len(gts)
        sorted_p = sorted(preds, key=lambda x: x['score'], reverse=True)
        for p in sorted_p:
            best_iou, best_idx = -1, -1
            for i, g in enumerate(gts):
                if not matched_gt[i] and p['category_id'] == g['category_id']:
                    iou = bbox_iou(p['bbox'], g['bbox'])
                    if iou > best_iou: best_iou, best_idx = iou, i
            if best_iou >= 0.5:
                f_tp += 1
                matched_gt[best_idx] = True
            else: f_fp += 1
        f_fn += matched_gt.count(False)

        # Store for mAP and Small Object metrics
        for p in preds: all_preds_cls[p['category_id']].append(p)
        for g in gts:
            all_gts_cls[g['category_id']].append(g)
            if g['area'] < SMALL_OBJ_THRESHOLD:
                # Local check for small recall
                best_iou = max([bbox_iou(p['bbox'], g['bbox']) for p in preds if p['category_id'] == g['category_id']] + [0])
                if best_iou >= 0.5: f_tp_s += 1
                else: f_fn_s += 1

        if idx < 3: # Save samples
            visual_result = visualize_object_predictions(np.array(read_image(image_path)), result.object_prediction_list)
            cv2.imwrite(os.path.join(OUTPUT_VIS_DIR, f"{folder}_{img_name}"), cv2.cvtColor(visual_result['image'], cv2.COLOR_RGB2BGR))

    # Fold Metrics
    p_f = f_tp / (f_tp + f_fp) if (f_tp + f_fp) > 0 else 0
    r_f = f_tp / (f_tp + f_fn) if (f_tp + f_fn) > 0 else 0
    f1_f = 2*(p_f*r_f)/(p_f+r_f) if (p_f+r_f)>0 else 0
    map50 = np.mean([calculate_ap(all_preds_cls[i], all_gts_cls[i]) for i in range(num_classes) if all_gts_cls[i]])
    s_rec = f_tp_s / (f_tp_s + f_fn_s) if (f_tp_s + f_fn_s) > 0 else 0

    results_log.append({
        'Folder': folder, 'mAP@0.5': round(map50, 4), 'Precision': round(p_f, 4),
        'Recall': round(r_f, 4), 'F1': round(f1_f, 4), 'Small_Recall': round(s_rec, 4)
    })

df = pd.DataFrame(results_log)
avg_row = df.mean(numeric_only=True).to_dict(); avg_row['Folder'] = 'AVERAGE'
df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
print("\n--- Final Results (SAHI + mAP) ---")
print(df.to_string(index=False))


--- 🚀 Starting Evaluation with mAP: fold_0 ---
Performing prediction on 3 slices.
Performing prediction on 3 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 4 slices.
Performing prediction on 1 slices.
Performing prediction on 6 slices.
Performing prediction on 20 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 8 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 3 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 4 slices.
Performing prediction on 20 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 9 slices.
Perfo

## Visualizations

####Subset: Full Dataset + Visualization

In [ ]:
import os
import cv2
import yaml
import numpy as np
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# ============================================================
# CONFIG
# ============================================================

BASE_PATH = "/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data"
DATA_YAML = "/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_0/data.yaml"

OUT_ROOT = "/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV26/full_dataset/tp_fp_fn_split"

FOLDS = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]

MODEL_MAP = {
    "fold_0": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/full_dataset/fold_0/weights/best.pt",
    "fold_1": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/full_dataset/fold_1/weights/best.pt",
    "fold_2": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/full_dataset/fold_2/weights/best.pt",
    "fold_3": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/full_dataset/fold_3/weights/best.pt",
    "fold_4": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/full_dataset/fold_4/weights/best.pt",
}

os.makedirs(OUT_ROOT, exist_ok=True)

# SAHI
SLICE = 640
OVERLAP = 0.2

CONF_MODEL = 0.30
CONF_LOW = 0.80
IOU_TH = 0.5

SAVE_N = 50  # images per fold

# ============================================================
# COLORS (BGR)
# ============================================================

MAGENTA = (255, 0, 255)
YELLOW  = (0, 255, 255)
RED     = (0, 0, 255)
ORANGE  = (0, 165, 255)
WHITE   = (255, 255, 255)
BLACK   = (0, 0, 0)

# ============================================================
# LOAD CLASS INFO
# ============================================================

with open(DATA_YAML, "r") as f:
    data_yaml = yaml.safe_load(f)

ID2NAME = {i: n for i, n in enumerate(data_yaml["names"])}

CONNECTOR_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "connector" in n.lower()
}

MISSING_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "missing" in n.lower()
}

print("Connector IDs:", CONNECTOR_CLASS_IDS)
print("Missing IDs:", MISSING_CLASS_IDS)

# ============================================================
# HELPERS
# ============================================================

def get_image_dimensions(path):
    with Image.open(path) as im:
        return im.size


def draw_text(img, text, x, y):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, BLACK, 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, WHITE, 1, cv2.LINE_AA)


def bbox_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter == 0:
        return 0.0

    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])

    return inter / (areaA + areaB - inter + 1e-9)


def load_gt(label_dir, img_name, dims):
    gts = []

    path = os.path.join(
        label_dir,
        os.path.splitext(img_name)[0] + ".txt"
    )

    if not os.path.exists(path):
        return gts

    W, H = dims

    with open(path) as f:
        for line in f:
            c, cx, cy, w, h = map(float, line.split())

            x1 = (cx - w/2) * W
            y1 = (cy - h/2) * H
            x2 = (cx + w/2) * W
            y2 = (cy + h/2) * H

            gts.append({
                "category_id": int(c),
                "bbox": [x1, y1, x2, y2]
            })

    return gts


def match(preds, gts):

    preds = sorted(preds, key=lambda x: x["score"], reverse=True)
    used = [False]*len(gts)

    tp, fp = [], []

    for p in preds:

        best_iou = 0
        best_j = -1

        for j, g in enumerate(gts):

            if used[j]:
                continue

            if p["category_id"] != g["category_id"]:
                continue

            iou = bbox_iou(p["bbox"], g["bbox"])

            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= IOU_TH:
            used[best_j] = True
            tp.append(p)
        else:
            fp.append(p)

    fn = [gts[i] for i in range(len(gts)) if not used[i]]

    return tp, fp, fn


def class_color(cid, mode):

    if mode == "fp":
        return ORANGE

    if mode == "fn":
        return RED

    if cid in CONNECTOR_CLASS_IDS:
        return YELLOW

    return MAGENTA


def should_label(cid, score, mode):

    if cid in MISSING_CLASS_IDS:
        return True

    if mode in ["fp", "fn"]:
        return True

    if score is not None and score < CONF_LOW:
        return True

    return False


def draw_boxes(img, items, mode):

    for it in items:

        cid = it["category_id"]
        score = it.get("score")

        x1, y1, x2, y2 = map(int, it["bbox"])

        color = class_color(cid, mode)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        if should_label(cid, score, mode):

            name = ID2NAME[cid]

            if mode == "fp":
                text = f"FP {name}:{score:.2f}"
            elif mode == "fn":
                text = f"FN {name}"
            else:
                text = f"{name}:{score:.2f}"

            draw_text(img, text, x1, max(15, y1-5))


def save_views(img_path, tp, fp, fn, out_base):

    img = cv2.imread(img_path)

    tp_img = img.copy()
    fp_img = img.copy()
    fn_img = img.copy()

    draw_boxes(tp_img, tp, "tp")
    draw_boxes(fp_img, fp, "fp")
    draw_boxes(fn_img, fn, "fn")

    cv2.imwrite(out_base + "_TP.png", tp_img)
    cv2.imwrite(out_base + "_FP.png", fp_img)
    cv2.imwrite(out_base + "_FN.png", fn_img)


# ============================================================
# MAIN LOOP
# ============================================================

for fold in FOLDS:

    print(f"\n🚀 Processing {fold}")

    img_dir = f"{BASE_PATH}/{fold}/valid/images"
    label_dir = f"{BASE_PATH}/{fold}/valid/labels"

    out_dir = os.path.join(OUT_ROOT, fold)
    os.makedirs(out_dir, exist_ok=True)

    model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=MODEL_MAP[fold],
        confidence_threshold=CONF_MODEL,
        device="cuda:0"
    )

    images = sorted(os.listdir(img_dir))[:SAVE_N]

    for img_name in images:

        img_path = os.path.join(img_dir, img_name)

        dims = get_image_dimensions(img_path)
        gts = load_gt(label_dir, img_name, dims)

        result = get_sliced_prediction(
            img_path,
            model,
            slice_height=SLICE,
            slice_width=SLICE,
            overlap_height_ratio=OVERLAP,
            overlap_width_ratio=OVERLAP,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.5
        )

        preds = []

        for obj in result.object_prediction_list:

            x1, y1, x2, y2 = obj.bbox.to_xyxy()

            preds.append({
                "category_id": int(obj.category.id),
                "score": float(obj.score.value),
                "bbox": [x1, y1, x2, y2]
            })

        tp, fp, fn = match(preds, gts)

        out_base = os.path.join(
            out_dir,
            os.path.splitext(img_name)[0]
        )

        save_views(img_path, tp, fp, fn, out_base)

    print(f"✅ Saved to {out_dir}")

print("\nDONE ✅")


####Subset: Components Only  + Visualization

In [ ]:
import os
import cv2
import yaml
import numpy as np
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# ============================================================
# CONFIG
# ============================================================

BASE_PATH = "/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data"
DATA_YAML = "/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_0/data.yaml"

OUT_ROOT = "/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV26/components_only/tp_fp_fn_split"

FOLDS = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]

MODEL_MAP = {
    "fold_0": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/components_only/fold_0/weights/best.pt",
    "fold_1": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/components_only/fold_1/weights/best.pt",
    "fold_2": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/components_only/fold_2/weights/best.pt",
    "fold_3": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/components_only/fold_3/weights/best.pt",
    "fold_4": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/components_only/fold_4/weights/best.pt",
}

os.makedirs(OUT_ROOT, exist_ok=True)

# SAHI
SLICE = 640
OVERLAP = 0.2

CONF_MODEL = 0.30
CONF_LOW = 0.80
IOU_TH = 0.5

SAVE_N = 50  # images per fold

# ============================================================
# COLORS (BGR)
# ============================================================

MAGENTA = (255, 0, 255)
YELLOW  = (0, 255, 255)
RED     = (0, 0, 255)
ORANGE  = (0, 165, 255)
WHITE   = (255, 255, 255)
BLACK   = (0, 0, 0)

# ============================================================
# LOAD CLASS INFO
# ============================================================

with open(DATA_YAML, "r") as f:
    data_yaml = yaml.safe_load(f)

ID2NAME = {i: n for i, n in enumerate(data_yaml["names"])}

CONNECTOR_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "connector" in n.lower()
}

MISSING_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "missing" in n.lower()
}

print("Connector IDs:", CONNECTOR_CLASS_IDS)
print("Missing IDs:", MISSING_CLASS_IDS)

# ============================================================
# HELPERS
# ============================================================

def get_image_dimensions(path):
    with Image.open(path) as im:
        return im.size


def draw_text(img, text, x, y):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, BLACK, 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, WHITE, 1, cv2.LINE_AA)


def bbox_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter == 0:
        return 0.0

    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])

    return inter / (areaA + areaB - inter + 1e-9)


def load_gt(label_dir, img_name, dims):
    gts = []

    path = os.path.join(
        label_dir,
        os.path.splitext(img_name)[0] + ".txt"
    )

    if not os.path.exists(path):
        return gts

    W, H = dims

    with open(path) as f:
        for line in f:
            c, cx, cy, w, h = map(float, line.split())

            x1 = (cx - w/2) * W
            y1 = (cy - h/2) * H
            x2 = (cx + w/2) * W
            y2 = (cy + h/2) * H

            gts.append({
                "category_id": int(c),
                "bbox": [x1, y1, x2, y2]
            })

    return gts


def match(preds, gts):

    preds = sorted(preds, key=lambda x: x["score"], reverse=True)
    used = [False]*len(gts)

    tp, fp = [], []

    for p in preds:

        best_iou = 0
        best_j = -1

        for j, g in enumerate(gts):

            if used[j]:
                continue

            if p["category_id"] != g["category_id"]:
                continue

            iou = bbox_iou(p["bbox"], g["bbox"])

            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= IOU_TH:
            used[best_j] = True
            tp.append(p)
        else:
            fp.append(p)

    fn = [gts[i] for i in range(len(gts)) if not used[i]]

    return tp, fp, fn


def class_color(cid, mode):

    if mode == "fp":
        return ORANGE

    if mode == "fn":
        return RED

    if cid in CONNECTOR_CLASS_IDS:
        return YELLOW

    return MAGENTA


def should_label(cid, score, mode):

    if cid in MISSING_CLASS_IDS:
        return True

    if mode in ["fp", "fn"]:
        return True

    if score is not None and score < CONF_LOW:
        return True

    return False


def draw_boxes(img, items, mode):

    for it in items:

        cid = it["category_id"]
        score = it.get("score")

        x1, y1, x2, y2 = map(int, it["bbox"])

        color = class_color(cid, mode)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        if should_label(cid, score, mode):

            name = ID2NAME[cid]

            if mode == "fp":
                text = f"FP {name}:{score:.2f}"
            elif mode == "fn":
                text = f"FN {name}"
            else:
                text = f"{name}:{score:.2f}"

            draw_text(img, text, x1, max(15, y1-5))


def save_views(img_path, tp, fp, fn, out_base):

    img = cv2.imread(img_path)

    tp_img = img.copy()
    fp_img = img.copy()
    fn_img = img.copy()

    draw_boxes(tp_img, tp, "tp")
    draw_boxes(fp_img, fp, "fp")
    draw_boxes(fn_img, fn, "fn")

    cv2.imwrite(out_base + "_TP.png", tp_img)
    cv2.imwrite(out_base + "_FP.png", fp_img)
    cv2.imwrite(out_base + "_FN.png", fn_img)


# ============================================================
# MAIN LOOP
# ============================================================

for fold in FOLDS:

    print(f"\n🚀 Processing {fold}")

    img_dir = f"{BASE_PATH}/{fold}/valid/images"
    label_dir = f"{BASE_PATH}/{fold}/valid/labels"

    out_dir = os.path.join(OUT_ROOT, fold)
    os.makedirs(out_dir, exist_ok=True)

    model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=MODEL_MAP[fold],
        confidence_threshold=CONF_MODEL,
        device="cuda:0"
    )

    images = sorted(os.listdir(img_dir))[:SAVE_N]

    for img_name in images:

        img_path = os.path.join(img_dir, img_name)

        dims = get_image_dimensions(img_path)
        gts = load_gt(label_dir, img_name, dims)

        result = get_sliced_prediction(
            img_path,
            model,
            slice_height=SLICE,
            slice_width=SLICE,
            overlap_height_ratio=OVERLAP,
            overlap_width_ratio=OVERLAP,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.5
        )

        preds = []

        for obj in result.object_prediction_list:

            x1, y1, x2, y2 = obj.bbox.to_xyxy()

            preds.append({
                "category_id": int(obj.category.id),
                "score": float(obj.score.value),
                "bbox": [x1, y1, x2, y2]
            })

        tp, fp, fn = match(preds, gts)

        out_base = os.path.join(
            out_dir,
            os.path.splitext(img_name)[0]
        )

        save_views(img_path, tp, fp, fn, out_base)

    print(f"✅ Saved to {out_dir}")

print("\nDONE ✅")


####Subset: Missing Only  + Visualization

In [ ]:
import os
import cv2
import yaml
import numpy as np
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# ============================================================
# CONFIG
# ============================================================

BASE_PATH = "/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data"
DATA_YAML = "/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/data.yaml"

OUT_ROOT = "/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV26/missing_only/tp_fp_fn_split"

FOLDS = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]

MODEL_MAP = {
    "fold_0": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/missing_only/fold_0/weights/best.pt",
    "fold_1": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/missing_only/fold_1/weights/best.pt",
    "fold_2": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/missing_only/fold_2/weights/best.pt",
    "fold_3": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/missing_only/fold_3/weights/best.pt",
    "fold_4": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/missing_only/fold_4/weights/best.pt",
}

os.makedirs(OUT_ROOT, exist_ok=True)

# SAHI
SLICE = 640
OVERLAP = 0.2

CONF_MODEL = 0.30
CONF_LOW = 0.80
IOU_TH = 0.5

SAVE_N = 50  # images per fold

# ============================================================
# COLORS (BGR)
# ============================================================

MAGENTA = (255, 0, 255)
YELLOW  = (0, 255, 255)
RED     = (0, 0, 255)
ORANGE  = (0, 165, 255)
WHITE   = (255, 255, 255)
BLACK   = (0, 0, 0)

# ============================================================
# LOAD CLASS INFO
# ============================================================

with open(DATA_YAML, "r") as f:
    data_yaml = yaml.safe_load(f)

ID2NAME = {i: n for i, n in enumerate(data_yaml["names"])}

CONNECTOR_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "connector" in n.lower()
}

MISSING_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "missing" in n.lower()
}

print("Connector IDs:", CONNECTOR_CLASS_IDS)
print("Missing IDs:", MISSING_CLASS_IDS)

# ============================================================
# HELPERS
# ============================================================

def get_image_dimensions(path):
    with Image.open(path) as im:
        return im.size


def draw_text(img, text, x, y):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, BLACK, 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, WHITE, 1, cv2.LINE_AA)


def bbox_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter == 0:
        return 0.0

    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])

    return inter / (areaA + areaB - inter + 1e-9)


def load_gt(label_dir, img_name, dims):
    gts = []

    path = os.path.join(
        label_dir,
        os.path.splitext(img_name)[0] + ".txt"
    )

    if not os.path.exists(path):
        return gts

    W, H = dims

    with open(path) as f:
        for line in f:
            c, cx, cy, w, h = map(float, line.split())

            x1 = (cx - w/2) * W
            y1 = (cy - h/2) * H
            x2 = (cx + w/2) * W
            y2 = (cy + h/2) * H

            gts.append({
                "category_id": int(c),
                "bbox": [x1, y1, x2, y2]
            })

    return gts


def match(preds, gts):

    preds = sorted(preds, key=lambda x: x["score"], reverse=True)
    used = [False]*len(gts)

    tp, fp = [], []

    for p in preds:

        best_iou = 0
        best_j = -1

        for j, g in enumerate(gts):

            if used[j]:
                continue

            if p["category_id"] != g["category_id"]:
                continue

            iou = bbox_iou(p["bbox"], g["bbox"])

            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= IOU_TH:
            used[best_j] = True
            tp.append(p)
        else:
            fp.append(p)

    fn = [gts[i] for i in range(len(gts)) if not used[i]]

    return tp, fp, fn


def class_color(cid, mode):

    if mode == "fp":
        return ORANGE

    if mode == "fn":
        return RED

    if cid in CONNECTOR_CLASS_IDS:
        return YELLOW

    return MAGENTA


def should_label(cid, score, mode):

    if cid in MISSING_CLASS_IDS:
        return True

    if mode in ["fp", "fn"]:
        return True

    if score is not None and score < CONF_LOW:
        return True

    return False


def draw_boxes(img, items, mode):

    for it in items:

        cid = it["category_id"]
        score = it.get("score")

        x1, y1, x2, y2 = map(int, it["bbox"])

        color = class_color(cid, mode)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        if should_label(cid, score, mode):

            name = ID2NAME[cid]

            if mode == "fp":
                text = f"FP {name}:{score:.2f}"
            elif mode == "fn":
                text = f"FN {name}"
            else:
                text = f"{name}:{score:.2f}"

            draw_text(img, text, x1, max(15, y1-5))


def save_views(img_path, tp, fp, fn, out_base):

    img = cv2.imread(img_path)

    tp_img = img.copy()
    fp_img = img.copy()
    fn_img = img.copy()

    draw_boxes(tp_img, tp, "tp")
    draw_boxes(fp_img, fp, "fp")
    draw_boxes(fn_img, fn, "fn")

    cv2.imwrite(out_base + "_TP.png", tp_img)
    cv2.imwrite(out_base + "_FP.png", fp_img)
    cv2.imwrite(out_base + "_FN.png", fn_img)


# ============================================================
# MAIN LOOP
# ============================================================

for fold in FOLDS:

    print(f"\n🚀 Processing {fold}")

    img_dir = f"{BASE_PATH}/{fold}/valid/images"
    label_dir = f"{BASE_PATH}/{fold}/valid/labels"

    out_dir = os.path.join(OUT_ROOT, fold)
    os.makedirs(out_dir, exist_ok=True)

    model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=MODEL_MAP[fold],
        confidence_threshold=CONF_MODEL,
        device="cuda:0"
    )

    images = sorted(os.listdir(img_dir))[:SAVE_N]

    for img_name in images:

        img_path = os.path.join(img_dir, img_name)

        dims = get_image_dimensions(img_path)
        gts = load_gt(label_dir, img_name, dims)

        result = get_sliced_prediction(
            img_path,
            model,
            slice_height=SLICE,
            slice_width=SLICE,
            overlap_height_ratio=OVERLAP,
            overlap_width_ratio=OVERLAP,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.5
        )

        preds = []

        for obj in result.object_prediction_list:

            x1, y1, x2, y2 = obj.bbox.to_xyxy()

            preds.append({
                "category_id": int(obj.category.id),
                "score": float(obj.score.value),
                "bbox": [x1, y1, x2, y2]
            })

        tp, fp, fn = match(preds, gts)

        out_base = os.path.join(
            out_dir,
            os.path.splitext(img_name)[0]
        )

        save_views(img_path, tp, fp, fn, out_base)

    print(f"✅ Saved to {out_dir}")

print("\nDONE ✅")


####Subset: Non Missing  + Visualization

In [ ]:
import os
import cv2
import yaml
import numpy as np
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# ============================================================
# CONFIG
# ============================================================

BASE_PATH = "/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data"
DATA_YAML = "/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_0/data.yaml"

OUT_ROOT = "/content/drive/MyDrive/PCB_MC/Results/SAHI/YOLOV26/non_missing/tp_fp_fn_split"

FOLDS = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]

MODEL_MAP = {
    "fold_0": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/non_missing/fold_0/weights/best.pt",
    "fold_1": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/non_missing/fold_1/weights/best.pt",
    "fold_2": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/non_missing/fold_2/weights/best.pt",
    "fold_3": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/non_missing/fold_3/weights/best.pt",
    "fold_4": "/content/drive/MyDrive/PCB_MC/Results/YOLOV26/non_missing/fold_4/weights/best.pt",
}

os.makedirs(OUT_ROOT, exist_ok=True)

# SAHI
SLICE = 640
OVERLAP = 0.2

CONF_MODEL = 0.30
CONF_LOW = 0.80
IOU_TH = 0.5

SAVE_N = 50  # images per fold

# ============================================================
# COLORS (BGR)
# ============================================================

MAGENTA = (255, 0, 255)
YELLOW  = (0, 255, 255)
RED     = (0, 0, 255)
ORANGE  = (0, 165, 255)
WHITE   = (255, 255, 255)
BLACK   = (0, 0, 0)

# ============================================================
# LOAD CLASS INFO
# ============================================================

with open(DATA_YAML, "r") as f:
    data_yaml = yaml.safe_load(f)

ID2NAME = {i: n for i, n in enumerate(data_yaml["names"])}

CONNECTOR_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "connector" in n.lower()
}

MISSING_CLASS_IDS = {
    i for i, n in ID2NAME.items()
    if "missing" in n.lower()
}

print("Connector IDs:", CONNECTOR_CLASS_IDS)
print("Missing IDs:", MISSING_CLASS_IDS)

# ============================================================
# HELPERS
# ============================================================

def get_image_dimensions(path):
    with Image.open(path) as im:
        return im.size


def draw_text(img, text, x, y):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, BLACK, 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, WHITE, 1, cv2.LINE_AA)


def bbox_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter == 0:
        return 0.0

    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])

    return inter / (areaA + areaB - inter + 1e-9)


def load_gt(label_dir, img_name, dims):
    gts = []

    path = os.path.join(
        label_dir,
        os.path.splitext(img_name)[0] + ".txt"
    )

    if not os.path.exists(path):
        return gts

    W, H = dims

    with open(path) as f:
        for line in f:
            c, cx, cy, w, h = map(float, line.split())

            x1 = (cx - w/2) * W
            y1 = (cy - h/2) * H
            x2 = (cx + w/2) * W
            y2 = (cy + h/2) * H

            gts.append({
                "category_id": int(c),
                "bbox": [x1, y1, x2, y2]
            })

    return gts


def match(preds, gts):

    preds = sorted(preds, key=lambda x: x["score"], reverse=True)
    used = [False]*len(gts)

    tp, fp = [], []

    for p in preds:

        best_iou = 0
        best_j = -1

        for j, g in enumerate(gts):

            if used[j]:
                continue

            if p["category_id"] != g["category_id"]:
                continue

            iou = bbox_iou(p["bbox"], g["bbox"])

            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= IOU_TH:
            used[best_j] = True
            tp.append(p)
        else:
            fp.append(p)

    fn = [gts[i] for i in range(len(gts)) if not used[i]]

    return tp, fp, fn


def class_color(cid, mode):

    if mode == "fp":
        return ORANGE

    if mode == "fn":
        return RED

    if cid in CONNECTOR_CLASS_IDS:
        return YELLOW

    return MAGENTA


def should_label(cid, score, mode):

    if cid in MISSING_CLASS_IDS:
        return True

    if mode in ["fp", "fn"]:
        return True

    if score is not None and score < CONF_LOW:
        return True

    return False


def draw_boxes(img, items, mode):

    for it in items:

        cid = it["category_id"]
        score = it.get("score")

        x1, y1, x2, y2 = map(int, it["bbox"])

        color = class_color(cid, mode)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        if should_label(cid, score, mode):

            name = ID2NAME[cid]

            if mode == "fp":
                text = f"FP {name}:{score:.2f}"
            elif mode == "fn":
                text = f"FN {name}"
            else:
                text = f"{name}:{score:.2f}"

            draw_text(img, text, x1, max(15, y1-5))


def save_views(img_path, tp, fp, fn, out_base):

    img = cv2.imread(img_path)

    tp_img = img.copy()
    fp_img = img.copy()
    fn_img = img.copy()

    draw_boxes(tp_img, tp, "tp")
    draw_boxes(fp_img, fp, "fp")
    draw_boxes(fn_img, fn, "fn")

    cv2.imwrite(out_base + "_TP.png", tp_img)
    cv2.imwrite(out_base + "_FP.png", fp_img)
    cv2.imwrite(out_base + "_FN.png", fn_img)


# ============================================================
# MAIN LOOP
# ============================================================

for fold in FOLDS:

    print(f"\n🚀 Processing {fold}")

    img_dir = f"{BASE_PATH}/{fold}/valid/images"
    label_dir = f"{BASE_PATH}/{fold}/valid/labels"

    out_dir = os.path.join(OUT_ROOT, fold)
    os.makedirs(out_dir, exist_ok=True)

    model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=MODEL_MAP[fold],
        confidence_threshold=CONF_MODEL,
        device="cuda:0"
    )

    images = sorted(os.listdir(img_dir))[:SAVE_N]

    for img_name in images:

        img_path = os.path.join(img_dir, img_name)

        dims = get_image_dimensions(img_path)
        gts = load_gt(label_dir, img_name, dims)

        result = get_sliced_prediction(
            img_path,
            model,
            slice_height=SLICE,
            slice_width=SLICE,
            overlap_height_ratio=OVERLAP,
            overlap_width_ratio=OVERLAP,
            postprocess_type="GREEDYNMM",
            postprocess_match_threshold=0.5
        )

        preds = []

        for obj in result.object_prediction_list:

            x1, y1, x2, y2 = obj.bbox.to_xyxy()

            preds.append({
                "category_id": int(obj.category.id),
                "score": float(obj.score.value),
                "bbox": [x1, y1, x2, y2]
            })

        tp, fp, fn = match(preds, gts)

        out_base = os.path.join(
            out_dir,
            os.path.splitext(img_name)[0]
        )

        save_views(img_path, tp, fp, fn, out_base)

    print(f"✅ Saved to {out_dir}")

print("\nDONE ✅")
